# [1.2] Intro to Mechanistic Interpretability: NNsight & Induction Circuits (exercises)

*Based on [ARENA 3.0](https://arena.education) by Callum McDougall.*
*Adapted for NNsight + nnterp by the NDIF team.*

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-12.png" width="350">

# Introduction

These pages are designed to get you introduced to the core concepts of mechanistic interpretability, via **NNsight** — a library for interpreting and manipulating the internals of neural networks.

Most of the sections are constructed in the following way:

1. A particular feature of NNsight is introduced.
2. You are given an exercise, in which you have to apply the feature.

The running theme of the exercises is **induction circuits**. Induction circuits are a particular type of circuit in a transformer, which can perform basic in-context learning. You should read the [corresponding section of Neel's glossary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=_Jzi6YHRHKP1JziwdE02qdYZ), before continuing. This [LessWrong post](https://www.lesswrong.com/posts/TvrfY4c9eaGLeyDkE/induction-heads-illustrated) might also help; it contains some diagrams (like the one below) which walk through the induction mechanism step by step.

Each exercise will have a difficulty and importance rating out of 5, as well as an estimated maximum time you should spend on these exercises and sometimes a short annotation. You should interpret the ratings & time estimates relatively (e.g. if you find yourself spending about 50% longer on the exercises than the time estimates, adjust accordingly). Please do skip exercises / look at solutions if you don't feel like they're important enough to be worth doing, and you'd rather get to the good stuff!

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram.png" width="1000">

## Content & Learning Objectives

### 1️⃣ NNsight: Introduction

This section is designed to get you up to speed with the NNsight library. You'll learn how to load and run models, understand the trace context for accessing internal activations, and use `.save()` to capture intermediate values.

> ##### Learning Objectives
>
> - Load and run a model using NNsight's `LanguageModel` wrapper
> - Understand how to use the trace context and `.save()` to cache activations
> - Use the model's tokenizer to convert text to tokens, and vice versa
> - Use `circuitsvis` to visualise attention heads

### 2️⃣ Finding induction heads

Here, you'll learn about induction heads, how they work and why they are important. You'll also learn how to identify them from the characteristic induction head stripe in their attention patterns when the model input is a repeating sequence.

> ##### Learning Objectives
>
> - Understand what induction heads are, and the algorithm they are implementing
> - Inspect activation patterns to identify basic attention head patterns, and write your own functions to detect attention heads for you
> - Identify induction heads by looking at the attention patterns produced from a repeating random sequence

### 3️⃣ NNsight: In-Trace Interventions & Logit Attribution

Next, you'll learn about in-trace interventions — NNsight's approach to accessing and modifying activations during a forward pass. You will also build some tools to perform logit attribution within your model, so you can identify which components are responsible for your model's performance on certain tasks.

> ##### Learning Objectives
>
> - Understand how NNsight's trace context allows accessing and intervening on activations
> - Use `.save()` to extract activations and process results
> - Build tools to perform attribution, i.e. detecting which components of your model are responsible for performance on a given task
> - Understand how in-trace interventions can be used to perform basic interventions like **ablation**

### 4️⃣ Reverse-engineering induction circuits

Lastly, these exercises show you how you can reverse-engineer a circuit by looking directly at a transformer's weights. You'll examine QK and OV circuits by multiplying through matrices using explicit einsums. You'll also look for evidence of composition between two induction heads, and once you've found it then you'll investigate the functionality of the full circuit formed from this composition.

> ##### Learning Objectives
>
> - Understand the difference between investigating a circuit by looking at activation patterns, and reverse-engineering a circuit by looking directly at the weights
> - Use explicit weight matrix multiplication to inspect the QK and OV circuits within an induction circuit
> - Perform further exploration of induction circuits: composition scores, and targeted ablations

### 5️⃣ Bonus: Introduction to nnterp

A brief introduction to the `nnterp` library, which provides a standardized interface for accessing transformer internals across different model architectures.

> ##### Learning Objectives
>
> - Understand how nnterp's `StandardizedTransformer` abstracts away architecture-specific details
> - See how the raw NNsight patterns you learned map to nnterp's cleaner API

## Setup code

In [1]:
import subprocess, sys

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/ndif-team/nnsight.git@v0.5.16"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "nnterp"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "circuitsvis", "plotly", "ipywidgets", "jaxtyping", "einops", "eindex-callum"])


In [2]:
import functools
import math
import sys
from pathlib import Path
from typing import Callable

import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F
from eindex import eindex
from IPython.display import display
from jaxtyping import Float, Int
from nnsight import LanguageModel, NNsight
from nnterp import StandardizedTransformer
from nnterp.rename_utils import RenameConfig, AttnProbFunction
from torch import Tensor
from tqdm import tqdm
from transformers import AutoTokenizer

# Add parent dir for plotly_utils and _conversion
# In Colab, you would clone the repo and adjust paths accordingly
arena_ch1_dir = Path(".").resolve().parent
sys.path.insert(0, str(arena_ch1_dir))
sys.path.insert(0, str(arena_ch1_dir / "_conversion"))

from convert_2L_attn_only import AttnOnly2L, load_model
from plotly_utils import (
    hist,
    imshow,
    plot_comp_scores,
    plot_logit_attribution,
    plot_loss_difference,
    to_numpy,
)

device = t.device("cuda" if t.cuda.is_available() else "cpu")

MAIN = __name__ == "__main__"


In [3]:
class FakeConfig:
    """nnterp calls model.config with __contains__ checks."""
    def __init__(self, **kwargs):
        for k, v in kwargs.items(): setattr(self, k, v)
    def __contains__(self, item): return hasattr(self, item)
    def __getitem__(self, item): return getattr(self, item)


class SourceSoftmaxAttnProbFunction(AttnProbFunction):
    """Accesses F.softmax() output inside any custom Attention.forward()."""
    def get_attention_prob_source(self, attention_module, return_module_source=False):
        if return_module_source: return attention_module.source
        return attention_module.source.F_softmax_0


# 1️⃣ NNsight: Introduction

> ##### Learning Objectives
>
> - Load and run a model using NNsight's `LanguageModel` wrapper
> - Understand how to use the trace context and `.save()` to cache activations
> - Use the model's tokenizer to convert text to tokens, and vice versa
> - Use `circuitsvis` to visualise attention heads

## Introduction

*Note - the conceptual material below is adapted from Neel Nanda's TransformerLens demo.*

This notebook introduces **[NNsight](https://nnsight.net)**, a library for doing [mechanistic interpretability](https://distill.pub/2020/circuits/zoom-in/) of neural networks. The goal of mechanistic interpretability is to take a trained model and reverse engineer the algorithms the model learned during training from its weights. It is a fact about the world today that we have computer programs that can essentially speak English at a human level (GPT-3, PaLM, etc), yet we have no idea how they work nor how to write one ourselves. This offends me greatly, and I would like to solve this! Mechanistic interpretability is a very young and small field, and there are a *lot* of open problems - if you would like to help, please try working on one! **Check out Neel Nanda's [list of concrete open problems](https://docs.google.com/document/d/1WONBzNqfKIxERejrrPlQMyKqg7jSFW92x5UMXNrMdPo/edit#) to figure out where to start.**

NNsight provides a clean interface for accessing and manipulating a model's internal activations during a forward pass, using a **trace context** and the `.save()` method. Unlike libraries that require custom model architectures, NNsight works with any PyTorch model — including standard HuggingFace transformers — by wrapping them in a `LanguageModel` (or more generally `NNsight`) object.

The core design principle is to enable exploratory analysis - one of the most fun parts of mechanistic interpretability compared to normal ML is the extremely short feedback loops! The point of this library is to keep the gap between having an experiment idea and seeing the results as small as possible, to make it easy for **research to feel like play** and to enter a flow state. For a great example of exploratory research in action, check out [Neel's notebook analysing Indirect Object Identification](https://github.com/neelnanda-io/TransformerLens/blob/main/Exploratory_Analysis_Demo.ipynb) or [his recording of himself doing research](https://www.youtube.com/watch?v=yo4QvDn-vsU)!

<details>
<summary>Original TransformerLens introduction (by Neel Nanda)</summary>

This is a demo notebook for [TransformerLens](https://github.com/neelnanda-io/TransformerLens), **a library I ([Neel Nanda](neelnanda.io)) wrote for doing [mechanistic interpretability](https://distill.pub/2020/circuits/zoom-in/) of GPT-2 Style language models.**

I wrote this library because after I left the Anthropic interpretability team and started doing independent research, I got extremely frustrated by the state of open source tooling. There's a lot of excellent infrastructure like HuggingFace and DeepSpeed to *use* or *train* models, but very little to dig into their internals and reverse engineer how they work. **This library tries to solve that**, and to make it easy to get into the field even if you don't work at an industry org with real infrastructure! The core features were heavily inspired by [Anthropic's excellent Garcon tool](https://transformer-circuits.pub/2021/garcon/index.html). Credit to Nelson Elhage and Chris Olah for building Garcon and showing me the value of good infrastructure for accelerating exploratory research!
</details>

## Loading and Running Models

NNsight's `LanguageModel` class wraps a HuggingFace model and its tokenizer together, making it easy to load pretrained models. For this demo we'll look at GPT-2 Small, an 80M parameter model.

```python
gpt2_small = LanguageModel("openai-community/gpt2", device_map="auto", dtype=torch.float32)
```

Under the hood, this downloads the GPT-2 model from HuggingFace and wraps it so we can access its internals via NNsight's trace context.

In [4]:
gpt2_small = StandardizedTransformer("openai-community/gpt2", device_map="auto", dtype=t.float32)


### Model Configuration

When wrapping models with nnterp's `StandardizedTransformer`, you provide a `RenameConfig` that maps your model's module names to nnterp's standardized API. For custom models (like our attention-only 2L model), we also attach a `FakeConfig` object so nnterp can read attributes like `num_attention_heads` and `hidden_size`.

You can always access the underlying model through `model._model`, and its configuration through `model._model.config` (or directly via the attributes you set).

### Exploring the model structure

You can inspect the model's architecture using `print(gpt2_small)` or by examining the underlying `_model` attribute. For GPT-2 Small, the key facts are:

* **12 layers** (each containing an attention block and an MLP block)
* **12 attention heads** per layer
* **768-dimensional** residual stream (`d_model = 768`)
* **64-dimensional** head size (`d_head = 64`)
* **Maximum context window** of 1024 tokens

The model's module tree follows HuggingFace's GPT-2 structure:
- `gpt2_small.layers[i]` — the i-th transformer block
- `gpt2_small.layers[i].attn` — the attention module
- `gpt2_small.layers[i].self_attn.c_attn` — the combined QKV projection
- `gpt2_small.layers[i].mlp` — the MLP module
- `gpt2_small.lm_head` — the final unembedding layer

### Running your model with NNsight's trace context

To run a model and access its internals, we use NNsight's **trace context**. Inside a `with model.trace(input):` block, we can access any module's output (or input) and call `.save()` to capture it. The saved values become available after the trace context exits.

```python
with gpt2_small.trace(tokens):
    logits = gpt2_small.logits.save()
# After the block, logits contains the actual tensor
```

This is analogous to TransformerLens's `run_with_cache`, but more flexible — you only save what you need, and you can access any module in the model's computational graph.

<details>
<summary>Aside — comparison with TransformerLens</summary>

In TransformerLens, you would use `logits, cache = model.run_with_cache(tokens)` to get all activations at once, stored in an `ActivationCache` object. NNsight's approach is more selective: you explicitly `.save()` only the activations you need, within a trace context. This is more memory-efficient and works with any PyTorch model, not just specially-designed ones.

TransformerLens also provides convenient shortcuts like `model.W_Q` to access all query weights at once, and reshapes attention weights to have separate `[head_index, d_model, d_head]` dimensions. In NNsight with HuggingFace models, you access the raw weights (e.g., GPT-2's concatenated `c_attn` weight) and reshape them yourself. This is more work but teaches you the actual model architecture.
</details>

In [5]:
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model the model out, let's find the loss on this paragraph!"""

tokens = gpt2_small.tokenizer(model_description_text, return_tensors="pt").input_ids.to(device)
with gpt2_small.trace(tokens):
    logits = gpt2_small.logits.save()
logits = logits.squeeze(0)  # [seq, vocab]

# Compute cross-entropy loss
loss = F.cross_entropy(logits[:-1], tokens.squeeze()[1:])
print("Model loss:", loss.item())


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loss: 4.347407341003418


<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Model loss: tensor(4.3443, device='cuda:0')</pre>

## Transformer architecture

Our 2-layer attention-only model follows a standard GPT-2-style architecture. The key details of its internal structure are:

* The weights `W_K`, `W_Q`, `W_V` mapping the residual stream to queries, keys and values are stored as separate matrices per head, with shape `[n_heads, d_model, d_head]`.
* `W_O` has shape `[n_heads, d_head, d_model]`, projecting each head's output back to the residual stream.
* **Important - we follow the convention that weight matrices multiply on the right rather than the left.** In other words, they have shape `[input, output]`, and we have `new_activation = old_activation @ weights + bias`.
    * Click the dropdown below for examples of this, if it seems unintuitive.

<details>
<summary>Examples of matrix multiplication in our model</summary>

* **Query matrices**
    * Each query matrix `W_Q` for a particular layer and head has shape `[d_model, d_head]`.
    * So if a vector `x` in the residual stream has length `d_model`, then the corresponding query vector is `x @ W_Q`, which has length `d_head`.
* **Embedding matrix**
    * The embedding matrix `W_E` has shape `[d_vocab, d_model]`.
    * So if `A` is a one-hot-encoded vector of length `d_vocab` corresponding to a particular token, then the embedding vector for this token is `A @ W_E`, which has length `d_model`.

</details>

You can access weights directly through `model._model` (e.g., `model._model.blocks[0].attn.W_Q` for layer 0 query weights). For a cleaner reference implementation, see the model definition in `_conversion/convert_2L_attn_only.py`.

### Parameters and Activations

It's important to distinguish between parameters and activations in the model.

* **Parameters** are the weights and biases that are learned during training.
    * These don't change when the model input changes.
    * They can be accessed directly from the model, e.g. `model.W_E` for the embedding matrix.
* **Activations** are temporary numbers calculated during a forward pass, that are functions of the input.
    * We can think of these values as only existing for the duration of a single forward pass, and disappearing afterwards.
    * We can use hooks to access these values during a forward pass (more on hooks later), but it doesn't make sense to talk about a model's activations outside the context of some particular input.
    * Attention scores and patterns are activations (this is slightly non-intuitve because they're used in a matrix multiplication with another activation).

The link below shows a diagram of a single layer (called a `TransformerBlock`) for an attention-only model with no biases. Each box corresponds to an **activation** (and also tells you the name of the corresponding hook point, which we will eventually use to access those activations). The red text below each box tells you the shape of the activation (ignoring the batch dimension). Each arrow corresponds to an operation on an activation; where there are **parameters** involved these are labelled on the arrows.

[Link to diagram](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/small-merm.svg)

The next link is to a diagram of a `TransformerBlock` with full features (including biases, layernorms, and MLPs). Don't worry if not all of this makes sense at first - we'll return to some of the details later. As we work with these transformers, we'll get more comfortable with their architecture.

[Link to diagram](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/full-merm.svg)

A few shortcuts to make your lives easier when using these models:

* You can index weights like `W_Q` directly from the model via e.g. `model.blocks[0].attn.W_Q` (which gives you the `[nheads, d_model, d_head]` query weights for all heads in layer 0).
    * But an easier way is just to index with `model.W_Q`, which gives you the `[nlayers, nheads, d_model, d_head]` tensor containing **every** query weight in the model.
* Similarly, there exist shortcuts `model.W_E`, `model.W_U` and `model.W_pos` for the embeddings, unembeddings and positional embeddings respectively.
* With models containing MLP layers, you also have `model.W_in` and `model.W_out` for the linear layers.
* The same is true for all biases (e.g. `model.b_Q` for all query biases).

## Tokenization

The tokenizer is stored inside the model at `model.tokenizer`, and you can use it directly. Standard HuggingFace tokenizer methods are available:

* `model.tokenizer.tokenize(text)` — splits text into token strings
* `model.tokenizer(text, return_tensors="pt")` — returns token ids as a tensor
* `model.tokenizer.decode(token_ids)` — converts token ids back to a string
* `model.tokenizer.batch_decode(list_of_ids)` — decodes multiple sequences

<details>
<summary>Aside - <code>&lt;|endoftext|&gt;</code></summary>

A weirdness you may notice is that GPT-2's tokenizer uses `<|endoftext|>` (token index 50256) as the Beginning of Sequence (BOS), End of Sequence (EOS), and padding token — they're all the same token. When working with GPT-2, it's common to prepend this token to inputs, because transformers tend to treat the first token weirdly (attention patterns need to sum to 1, so heads that want to be "off" often attend to the first token). Giving them a BOS token lets the heads rest by looking at that, preserving information in the first "real" token.
</details>

In [6]:
print(gpt2_small.tokenizer.tokenize("gpt2"))
print(gpt2_small.tokenizer("gpt2", return_tensors="pt").input_ids)
print(gpt2_small.tokenizer.decode([50256, 70, 457, 17]))


['g', 'pt', '2']
tensor([[ 70, 457,  17]])
<|endoftext|>gpt2


<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">['<|endoftext|>', 'g', 'pt', '2']
[['<|endoftext|>', 'g', 'pt', '2'], ['<|endoftext|>', 'g', 'pt', '2']]
tensor([[50256,    70,   457,    17]], device='cuda:0')
<|endoftext|>gpt2</pre>

<details>
<summary>Aside - <code>&lt;|endoftext|&gt;</code></summary>

A weirdness you may have noticed in the above is that tokenization added a `<|endoftext|>` to the start of each prompt. This is the **Beginning of Sequence (BOS)** token (which for GPT-2 is also the same as the EOS and PAD tokens - index `50256`).

Our tokenizer prepends this token by default. Notably, this happens whenever you call `tokenizer(text, return_tensors="pt")`. You can disable this by passing `add_special_tokens=False`.

The reason BOS tokens are useful: transformers tend to treat the first token weirdly - this doesn't really matter in training (where all inputs are >1000 tokens), but this can be a big issue when investigating short prompts! The reason for this is that attention patterns are a probability distribution and so need to add up to one, so to simulate being "off" they normally look at the first token. Giving them a BOS token lets the heads rest by looking at that, preserving the information in the first "real" token.

Further, *some* models are trained to need a BOS token (OPT and some interpretability-friendly models are, GPT-2 and GPT-Neo are not). But despite GPT-2 not being trained with this, empirically it seems to make interpretability easier.

</details>

### Exercise - how many tokens does your model guess correctly?

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to ~10 minutes on this exercise.
> ```

Consider the `model_description_text` you fed into your model above. How many tokens did your model guess correctly? Which tokens were correct?

In [7]:
tokens = gpt2_small.tokenizer(model_description_text, return_tensors="pt").input_ids.to(device)
with gpt2_small.trace(tokens):
    logits = gpt2_small.logits.save()
logits = logits.squeeze(0)  # [seq, vocab]
prediction = logits.argmax(dim=-1)[:-1]

true_tokens = tokens.squeeze()[1:]
is_correct = prediction == true_tokens

print(f"Model accuracy: {is_correct.sum()}/{len(true_tokens)}")
print(f"Correct tokens: {gpt2_small.tokenizer.batch_decode(prediction[is_correct].unsqueeze(-1))}")


Model accuracy: 35/110
Correct tokens: ['\n', '\n', 'former', ' with', ' models', '.', ' can', ' of', ' them', 'ooked', 'Trans', 'former', '_', 'NAME', '`.', ' model', ' loaded', ' the', 'ed', 'Trans', 'former', ' to', ' be', ' and', '-', '.', '\n', '\n', ' at', 'PT', '-', ',', ',', "'s", ' the']


**Induction heads** are a special kind of attention head which we'll examine a lot more in coming exercises. They allow a model to perform in-context learning of a specific form: generalising from one observation that token `B` follows token `A`, to predict that token `B` will follow `A` in future occurrences of `A`, even if these two tokens had never appeared together in the model's training data.

**Can you see evidence of any induction heads at work, on this text?**

<details>
<summary>Evidence of induction heads</summary>

The evidence for induction heads comes from the fact that the model successfully predicted repeated subsequences in the text. When a distinctive multi-token sequence appeared for the second time, the model predicted the continuation correctly — having learned the pattern from the first occurrence earlier in the context.
</details>

<details>
<summary>Hint</summary>

Use `return_type="logits"` to get the model's predictions, then take argmax across the vocab dimension. Then, compare these predictions with the actual tokens, derived from the `model_description_text`.

Remember, you should be comparing the `[:-1]`th elements of this tensor of predictions with the `[1:]`th elements of the input tokens (because your model's output represents a probability distribution over the *next* token, not the current one).

Also, remember to handle the batch dimension (since `logits`, and the output of `to_tokens`, will both have batch dimensions by default).

</details>

<details>
<summary>Answer - what you should see</summary>

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Model accuracy: 33/111
Correct tokens: ['\n', '\n', 'former', ' with', ' models', '.', ' can', ' of', 'ooked', 'Trans', 'former', '_', 'NAME', '`.', ' model', ' the', 'Trans', 'former', ' to', ' be', ' and', '-', '.', '\n', '\n', ' at', 'PT', '-', ',', ' model', ',', "'s", ' the']
</pre>

So the model got 33 out of 111 tokens correct. Not too bad!

</details>


<details><summary>Solution</summary>

```python
logits: Tensor = gpt2_small(model_description_text, return_type="logits")
prediction = logits.argmax(dim=-1).squeeze()[:-1]

true_tokens = gpt2_small.to_tokens(model_description_text).squeeze()[1:]
is_correct = prediction == true_tokens

print(f"Model accuracy: {is_correct.sum()}/{len(true_tokens)}")
print(f"Correct tokens: {gpt2_small.to_str_tokens(prediction[is_correct])}")
```
</details>

## Caching Activations with NNsight

The first basic operation when doing mechanistic interpretability is to break open the black box of the model and look at all of the internal activations of a model. In NNsight, this is done using the **trace context** with `.save()`:

```python
with model.trace(tokens):
    activation = model.transformer.h[0].self_attn.c_attn.output.save()
```

This runs the model on the given tokens and saves the output of any module you specify. You can save as many activations as you want within a single trace.

<details>
<summary>Aside — comparison with TransformerLens's <code>run_with_cache</code></summary>

In TransformerLens, `logits, cache = model.run_with_cache(tokens)` saves *every* activation at every hook point in the model. This can use a lot of memory for large models. With NNsight, you selectively `.save()` only the activations you need, which is more memory-efficient. The tradeoff is that you need to know which module paths to access (e.g., `model.transformer.h[0].self_attn.c_attn.output`), rather than using convenient shorthand keys like `cache["pattern", 0]`.
</details>

<details>
<summary>Aside - a note on <code>remove_batch_dim</code> (TransformerLens)</summary>

In TransformerLens, every activation inside the model begins with a batch dimension. When you only enter a single batch dimension, that dimension is always length 1 and kinda annoying, so passing in the `remove_batch_dim=True` keyword removes it. In NNsight, you can simply index with `[0]` to remove the batch dimension when needed, or use `.squeeze(0)`.
</details>

In [8]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.tokenizer(gpt2_text, return_tensors="pt").input_ids.to(device)

# Cache: run a trace and save everything we need
# NNsight requires accessing modules in forward-execution order
with gpt2_small.trace(gpt2_tokens):
    # Save attention internals for layer 0 (c_attn runs early in forward pass)
    gpt2_c_attn_out_0 = gpt2_small.layers[0].self_attn.c_attn.output.save()
    gpt2_logits = gpt2_small.logits.save()

print(type(gpt2_logits), gpt2_logits.shape)


<class 'torch.Tensor'> torch.Size([1, 32, 50257])


If you inspect the `gpt2_cache` object, you should see that it contains a very large number of keys, each one corresponding to a different activation in the model. You can access the keys by indexing the cache directly, or by a more convenient indexing shorthand. For instance, here are 2 ways to extract the attention patterns for layer 0:

<details>
<summary>Aside: <code>utils.get_act_name</code></summary>

The reason these are the same is that, under the hood, the first example actually indexes by `utils.get_act_name("pattern", 0)`, which evaluates to `"blocks.0.attn.hook_pattern"`.

In general, `utils.get_act_name` is a useful function for getting the full name of an activation, given its short name and layer number.

You can use the diagram from the **Transformer Architecture** section to help you find activation names.
</details>

### Exercise - verify activations

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Verify that the query, key, and attention pattern are related to each other in the expected way. Do this by extracting Q and K from the saved `c_attn` output (which concatenates Q, K, V along the last dimension), computing attention scores, and comparing to what you'd expect.

In GPT-2, the `c_attn` module projects the residual stream into a concatenated `[Q, K, V]` tensor of shape `[batch, seq, 3*d_model]`. You need to:
1. Split this into Q, K, V (each of shape `[batch, seq, d_model]`)
2. Reshape Q and K to `[seq, n_heads, d_head]`
3. Compute attention scores, apply causal mask, scale, and softmax

In [9]:
# Compute attention patterns from saved c_attn output
d_model = 768
n_heads = 12
d_head = d_model // n_heads

# c_attn output is [batch, seq, 3*d_model] = concatenated Q, K, V
qkv = gpt2_c_attn_out_0.squeeze(0)  # [seq, 3*d_model]
q, k, v = qkv.split(d_model, dim=-1)  # each [seq, d_model]

# Reshape to [seq, n_heads, d_head]
q = q.view(-1, n_heads, d_head)
k = k.view(-1, n_heads, d_head)

# Compute attention scores
seq_len = q.shape[0]
attn_scores = einops.einsum(q, k, "seqQ n h, seqK n h -> n seqQ seqK")
mask = t.triu(t.ones((seq_len, seq_len), dtype=t.bool, device=device), diagonal=1)
attn_scores.masked_fill_(mask, -1e9)
layer0_pattern = (attn_scores / d_head**0.5).softmax(-1)

print("Attention pattern shape:", layer0_pattern.shape)
print("Tests passed!")


Attention pattern shape: torch.Size([12, 32, 32])
Tests passed!


<details>
<summary>Hint</summary>

You'll need to use three different cache indexes in all:

* `gpt2_cache["pattern", 0]` to get the attention patterns, which have shape `[nhead, seqQ, seqK]`
* `gpt2_cache["q", 0]` to get the query vectors, which have shape `[seqQ, nhead, headsize]`
* `gpt2_cache["k", 0]` to get the key vectors, which have shape `[seqK, nhead, headsize]`

</details>


<details><summary>Solution</summary>

```python
layer0_pattern_from_cache = gpt2_cache["pattern", 0]

q, k = gpt2_cache["q", 0], gpt2_cache["k", 0]
seq, nhead, headsize = q.shape
layer0_attn_scores = einops.einsum(q, k, "seqQ n h, seqK n h -> n seqQ seqK")
mask = t.triu(t.ones((seq, seq), dtype=t.bool), diagonal=1).to(device)
layer0_attn_scores.masked_fill_(mask, -1e9)
layer0_pattern_from_q_and_k = (layer0_attn_scores / headsize**0.5).softmax(-1)
```
</details>

## Visualising Attention Heads

A key insight from the Mathematical Frameworks paper is that we should focus on interpreting the parts of the model that are intrinsically interpretable - the input tokens, the output logits and the attention patterns. Everything else (the residual stream, keys, queries, values, etc) are compressed intermediate states when calculating meaningful things. So a natural place to start is classifying heads by their attention patterns on various texts.

When doing interpretability, it's always good to begin by visualising your data, rather than taking summary statistics. Summary statistics can be super misleading! We'll use the `circuitsvis` library to visualise attention patterns.

In [10]:
gpt2_str_tokens = [gpt2_small.tokenizer.decode(t_) for t_ in gpt2_tokens.squeeze()]

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_patterns(
        tokens=gpt2_str_tokens,
        attention=layer0_pattern,
    )
)


Layer 0 Head Attention Patterns:


Hover over heads to see the attention patterns; click on a head to lock it. Hover over each token to see which other tokens it attends to (or which other tokens attend to it - you can see this by changing the dropdown to `Destination <- Source`).

<details>
<summary>Other circuitsvis functions - neuron activations</summary>

The `circuitsvis` library also has a number of cool visualisations for **neuron activations**. Here are some more of them (you don't have to understand them all now, but you can come back to them later).

The function below visualises neuron activations. The example shows just one sequence, but it can also show multiple sequences (if `tokens` is a list of lists of strings, and `activations` is a list of tensors).

```python
neuron_activations_for_all_layers = t.stack([
    gpt2_cache["post", layer] for layer in range(gpt2_small.cfg.n_layers)
], dim=1)
# shape = (seq_pos, layers, neurons)

cv.activations.text_neuron_activations(
    tokens=gpt2_str_tokens,
    activations=neuron_activations_for_all_layers
)
```

The next function shows which words each of the neurons activates most / least on (note that it requires some weird indexing to work correctly).

```python
neuron_activations_for_all_layers_rearranged = utils.to_numpy(einops.rearrange(neuron_activations_for_all_layers, "seq layers neurons -> 1 layers seq neurons"))

cv.topk_tokens.topk_tokens(
    # Some weird indexing required here ¯\_(ツ)_/¯
    tokens=[gpt2_str_tokens],
    activations=neuron_activations_for_all_layers_rearranged,
    max_k=7,
    first_dimension_name="Layer",
    third_dimension_name="Neuron",
    first_dimension_labels=list(range(12))
)
```
</details>

# 2️⃣ Finding induction heads

> ##### Learning Objectives
>
> - Understand what induction heads are, and the algorithm they are implementing
> - Inspect activation patterns to identify basic attention head patterns, and write your own functions to detect attention heads for you
> - Identify induction heads by looking at the attention patterns produced from a repeating random sequence

## Introducing Our Toy Attention-Only Model

Here we introduce a toy 2L attention-only transformer trained specifically for today. Some changes to make them easier to interpret:
- It has only attention blocks.
- The positional embeddings are only added to the residual stream before calculating each key and query vector in the attention layers as opposed to the token embeddings - i.e. we compute queries as `Q = (resid + pos_embed) @ W_Q + b_Q` and same for keys, but values as `V = resid @ W_V + b_V`. This means that **the residual stream can't directly encode positional information**.
    - This turns out to make it *way* easier for induction heads to form, it happens 2-3x times earlier - [see the comparison of two training runs](https://wandb.ai/mechanistic-interpretability/attn-only/reports/loss_ewma-22-08-24-22-08-00---VmlldzoyNTI2MDM0?accessToken=r6v951q0e1l4q4o70wb2q67wopdyo3v69kz54siuw7lwb4jz6u732vo56h6dr7c2).

We load this model using our custom `AttnOnly2L` class and wrap it with NNsight's `LanguageModel` for tracing capabilities.

In [11]:
# Load the converted 2L attention-only model
local_2l_path = arena_ch1_dir / "_conversion" / "attn_only_2L_converted.pt"
attn_2l = load_model(
    state_dict_path=str(local_2l_path) if local_2l_path.exists() else None,
    device=str(device),
)
attn_2l = attn_2l.to(device)
tokenizer_2l = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")

# Wrap with nnterp StandardizedTransformer for consistent accessors
attn_2l.config = FakeConfig(num_attention_heads=12, hidden_size=768, vocab_size=50278)
rename_config = RenameConfig(
    layers_name="blocks", attn_name="attn", mlp_name=None,
    ln_final_name=None, lm_head_name="unembed",
    attn_prob_source=SourceSoftmaxAttnProbFunction(),
    ignore_mlp=True,
)
model = StandardizedTransformer(
    attn_2l, rename_config=rename_config, tokenizer=tokenizer_2l,
    device_map=None, check_renaming=False,
)
model.attention_probabilities.enabled = True


Note that in the last section we had to define a tokenizer explicitly, and passed it into our model. But here, we just pass a tokenizer name, and the model will automatically create a tokenizer for us (under the hood, it calls `AutoTokenizer.from_pretrained(tokenizer_name)`).

Below, you'll load in your weights, with some boilerplate code to download your state dict from HuggingFace (you can do this for any model you've uploaded to HuggingFace yourself):

Finally, we'll create our model and load in the weights:

### Exercise - visualise & inspect attention patterns

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to ~10 minutes on this exercise.
> ```

Visualise the attention patterns for both layers of your model. We use nnterp's `attention_probabilities` accessor to extract attention patterns cleanly within a trace context.

The function `get_attention_patterns_2l` below runs a trace on the model, saving the attention probabilities at each layer. Inspect the results — what patterns do you notice?

In [12]:
def get_attention_patterns_2l(
    model: StandardizedTransformer, tokens_or_text
) -> list[Tensor]:
    """
    Get attention patterns for all layers/heads using nnterp's attention_probabilities accessor.

    Args:
        model: The nnterp-wrapped StandardizedTransformer
        tokens_or_text: Either a string or a tensor of token ids [batch, seq]

    Returns a list of tensors, one per layer, each of shape [n_heads, seq, seq]
    (batch dimension squeezed if batch_size=1) or [batch, n_heads, seq, seq].
    """
    if isinstance(tokens_or_text, str):
        tokens_or_text = model.tokenizer(tokens_or_text, return_tensors="pt").input_ids.to(device)

    n_layers = model._model.n_layers
    patterns = [None] * n_layers
    with model.trace(tokens_or_text):
        for i in range(n_layers):
            patterns[i] = model.attention_probabilities[i].save()

    # Squeeze batch dim if batch_size == 1
    return [p.squeeze(0) if p.shape[0] == 1 else p for p in patterns]


In [13]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

patterns = get_attention_patterns_2l(model, text)
str_tokens = [tokenizer_2l.decode(t_) for t_ in tokenizer_2l(text).input_ids]

for layer in range(model._model.n_layers):
    display(
        cv.attention.attention_patterns(
            tokens=str_tokens, attention=patterns[layer]
        )
    )


<details>
<summary>Discussion of results</summary>

We notice that there are three basic patterns which repeat quite frequently:

* `prev_token_heads`, which attend mainly to the previous token (e.g. head `0.7`)
* `current_token_heads`, which attend mainly to the current token (e.g. head `1.6`)
* `first_token_heads`, which attend mainly to the first token (e.g. heads `0.3`)

The `prev_token_heads` are important for the induction circuit, as we'll see shortly.
</details>

*(Note that we've run the model on the string `text`, rather than on tokens — our `StandardizedTransformer` wrapper supports both strings and pre-tokenized tensors as input.)*

Inspect the attention patterns. What do you notice about the attention heads?

You should spot three relatively distinctive basic patterns, which occur in multiple heads. What are these patterns, and can you guess why they might be present?

<details>
<summary>Aside - what to do if your plots won't show up</summary>

A common mistake is to fail to pass the tokens in as arguments. If you do this, your attention patterns won't render.

If this isn't the problem, then it might be an issue with the Circuitsvis library.Rather than plotting inline, you can do the following, and then open in your browser from the left-hand file explorer menu of VSCode:
</details>

<details>
<summary>Discussion of results </summary>

We notice that there are three basic patterns which repeat quite frequently:

* `prev_token_heads`, which attend mainly to the previous token (e.g. head `0.7`)
* `current_token_heads`, which attend mainly to the current token (e.g. head `1.6`)
* `first_token_heads`, which attend mainly to the first token (e.g. heads `0.3` or `1.4`, although these are a bit less clear-cut than the other two)

The `prev_token_heads` and `current_token_heads` are perhaps unsurprising, because words that are close together in a sequence probably have a lot more mutual information (i.e. we could get quite far using bigram or trigram prediction).

The `first_token_heads` are a bit more surprising. The basic intuition here is that the first token in a sequence is often used as a resting or null position for heads that only sometimes activate (since our attention probabilities always have to add up to 1).
</details>


<details><summary>Solution</summary>

```python
str_tokens = [tokenizer_2l.decode(t_) for t_ in tokenizer_2l(text).input_ids]
patterns = get_attention_patterns_2l(model, text)
for layer in range(model._model.n_layers):
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=patterns[layer]))
```
</details>

Now that we've observed our three basic attention patterns, it's time to make detectors for those patterns!

### Exercise - write your own detectors

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You shouldn't spend more than 10-25 minutes on these exercises.
> ```

You should fill in the functions below, which act as detectors for particular types of heads. Validate your detectors by comparing these results to the visual attention patterns above.

In [14]:
def current_attn_detector(patterns: list[Tensor]) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads.
    """
    attn_heads = []
    for layer in range(len(patterns)):
        for head in range(patterns[layer].shape[0]):
            attention_pattern = patterns[layer][head]
            score = attention_pattern.diagonal().mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def prev_attn_detector(patterns: list[Tensor]) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads.
    """
    attn_heads = []
    for layer in range(len(patterns)):
        for head in range(patterns[layer].shape[0]):
            attention_pattern = patterns[layer][head]
            score = attention_pattern.diagonal(-1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def first_attn_detector(patterns: list[Tensor]) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads.
    """
    attn_heads = []
    for layer in range(len(patterns)):
        for head in range(patterns[layer].shape[0]):
            attention_pattern = patterns[layer][head]
            score = attention_pattern[:, 0].mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


print("Heads attending to current token  = ", ", ".join(current_attn_detector(patterns)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(patterns)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(patterns)))


Heads attending to current token  =  
Heads attending to previous token =  0.7
Heads attending to first token    =  0.0, 0.1, 0.3, 0.5, 0.6, 0.10, 1.1


<details>
<summary>Hint</summary>

Try and compute the average attention probability along the relevant tokens. For instance, you can get the tokens just below the diagonal by using `t.diagonal` with appropriate `offset` parameter:

```python
def current_attn_detector(patterns: list[Tensor]) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads.
    """
    attn_heads = []
    for layer in range(len(patterns)):
        for head in range(patterns[layer].shape[0]):
            attention_pattern = patterns[layer][head]
            score = attention_pattern.diagonal().mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```

</details>

<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

```python
# take avg of diagonal elements
score = attention_pattern.diagonal().mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def prev_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of sub-diagonal elements
            score = attention_pattern.diagonal(-1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def first_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of 0th elements
            score = attention_pattern[:, 0].mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```

</details>


Compare the printouts to your attention visualisations above. Do they seem to make sense? As a bonus exercise, try inputting different text, and see how stable your results are. Do certain heads always get classified the same way?

## What are induction heads?

(Note: I use induction **head** to refer to the head in the second layer which attends to the 'token immediately after the copy of the current token', and induction **circuit** to refer to the circuit consisting of the composition of a **previous token head** in layer 0 and an **induction head** in layer 1)

[Induction heads](https://transformer-circuits.pub/2021/framework/index.html#induction-heads) are the first sophisticated circuit we see in transformers! And are sufficiently interesting that Anthropic wrote [another paper just about them](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html).

The induction circuit consists of:
1. A **previous token head** in an earlier layer, which copies information about the previous token into the current position's residual stream
2. An **induction head** in a later layer, which uses this information to identify positions where the current token appeared before, and then copies what came after it

For example, given the sequence `...The cat sat on the mat. The cat`, an induction head would predict `sat` by finding the previous occurrence of `The cat` and attending to `sat` (the token after `cat` in the first occurrence).

## Checking for the induction capability

A striking thing about models with induction heads is that, given a repeated sequence of random tokens, they can predict the repeated half of the sequence. This is nothing like its training data, so this is kind of wild!

To check that this model has induction heads, we're going to run it on exactly that, and compare performance on the two halves - you should see a striking difference in the per token losses.

## Checking for the induction capability

A striking thing about models with induction heads is that, given a repeated sequence of random tokens, they can predict the repeated half of the sequence. This is nothing like it's training data, so this is kind of wild! The ability to predict this kind of out of distribution generalisation is a strong point of evidence that you've really understood a circuit.

To check that this model has induction heads, we're going to run it on exactly that, and compare performance on the two halves - you should see a striking difference in the per token losses.

Note - we're using small sequences (and just one sequence), since the results are very obvious and this makes it easier to visualise. In practice we'd obviously use larger ones on more subtle tasks. But it's often easiest to iterate and debug on small tasks.

### Exercise - plot per-token loss on repeated sequence

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
>
> You shouldn't spend more than 10-15 minutes on these exercises.
> ```

Fill in the functions below to generate repeated token sequences and measure model performance.

In [15]:
def generate_repeated_tokens(
    model, seq_len: int, batch_size: int = 1
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Generates a sequence of repeated random tokens.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
    """
    t.manual_seed(0)
    bos_id = model.tokenizer.bos_token_id
    if bos_id is None:
        bos_id = model.tokenizer.eos_token_id
    d_vocab = model._model.d_vocab
    prefix = (t.ones(batch_size, 1) * bos_id).long()
    rep_tokens_half = t.randint(0, d_vocab, (batch_size, seq_len), dtype=t.int64)
    rep_tokens = t.cat([prefix, rep_tokens_half, rep_tokens_half], dim=-1).to(device)
    return rep_tokens


def run_and_cache_model_repeated_tokens(
    model: StandardizedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, list[Tensor]]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it,
    returning (tokens, logits, patterns) in a single trace.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        patterns: list of attention pattern tensors per layer
    """
    rep_tokens = generate_repeated_tokens(model, seq_len, batch_size)
    n_layers = model._model.n_layers

    saved_patterns = [None] * n_layers
    with model.trace(rep_tokens):
        for i in range(n_layers):
            saved_patterns[i] = model.attention_probabilities[i].save()
        rep_logits = model.lm_head.output.save()

    # Squeeze batch dim for patterns if batch=1
    patterns = [p.squeeze(0) if p.shape[0] == 1 else p for p in saved_patterns]
    return rep_tokens, rep_logits, patterns


def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    logprobs = logits.log_softmax(dim=-1)
    correct_logprobs = eindex(logprobs, tokens, "b s [b s+1]")
    return correct_logprobs


In [16]:
seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_patterns) = run_and_cache_model_repeated_tokens(
    model, seq_len, batch_size
)
# For single batch, squeeze patterns
rep_patterns_squeezed = [p.squeeze(0) if p.dim() == 4 else p for p in rep_patterns]
rep_str = [tokenizer_2l.decode(t_) for t_ in rep_tokens.squeeze()]
log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")

plot_loss_difference(log_probs, rep_str, seq_len)


/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:288: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  assert torch.tensor(shape).prod().item() == index_tensor[idx].numel(), \
/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:292: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  full_id

Performance on the first half: -24.912
Performance on the second half: -14.328


<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

<details>
<summary>Hint</summary>

You can define the first half of the repeated tokens using `t.randint(low, high, shape)`. Also remember to specify `dtype=t.long`.

Then you can concatenate together your prefix and two copies of the repeated tokens, using `t.concat`.
</details>


<details><summary>Solution</summary>

```python
def generate_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Generates a sequence of repeated random tokens

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
    """
    t.manual_seed(0)  # for reproducibility
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()
    rep_tokens_half = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.int64)
    rep_tokens = t.cat([prefix, rep_tokens_half, rep_tokens_half], dim=-1).to(device)
    return rep_tokens


def run_and_cache_model_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, ActivationCache]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it, returning (tokens,
    logits, cache). This function should use the `generate_repeated_tokens` function above.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        rep_cache: The cache of the model run on rep_tokens
    """
    rep_tokens = generate_repeated_tokens(model, seq_len, batch_size)
    rep_logits, rep_cache = model.run_with_cache(rep_tokens)
    return rep_tokens, rep_logits, rep_cache
```
</details>

</details>

### Looking for Induction Attention Patterns

The next natural thing to check for is the induction attention pattern. Let's visualise the attention patterns on our repeated sequence and look for the characteristic diagonal stripe.

In [17]:
for layer in range(model._model.n_layers):
    display(
        cv.attention.attention_patterns(
            tokens=rep_str, attention=rep_patterns_squeezed[layer]
        )
    )


<details>
<summary>Some observations</summary>

The characteristic pattern of induction heads is a diagonal stripe, with the diagonal offset as `seq_len-1` (because the destination token attends to the token *after* the destination token's previous occurrence).

You should see that heads 4 and 10 are strongly induction-y, head 6 is very weakly induction-y, and the rest aren't.
</details>

### Exercise - make an induction-head detector

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
>
> You shouldn't spend more than 5-15 minutes on this exercise.
> ```

Now, make an induction pattern score function, which looks for the average attention paid to the offset diagonal. Do this in the same style as our earlier head scorers.

Now, you should make an induction pattern score function, which looks for the average attention paid to the offset diagonal. Do this in the same style as our earlier head scorers, just with a different kind of indexing that is appropriate for detecting the characteristic attention head pattern.

In [18]:
def induction_attn_detector(patterns: list[Tensor]) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be induction heads.

    Remember - the tokens used to generate rep_cache are (bos_token, *rand_tokens, *rand_tokens)
    """
    attn_heads = []
    for layer in range(len(patterns)):
        for head in range(patterns[layer].shape[0]):
            attention_pattern = patterns[layer][head]
            seq_len = (attention_pattern.shape[-1] - 1) // 2
            score = attention_pattern.diagonal(-seq_len + 1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


if MAIN:
    print("Induction heads = ", ", ".join(induction_attn_detector(rep_patterns_squeezed)))


Induction heads =  1.4, 1.10


<details>
<summary>Help - I'm not sure what offset to use.</summary>

The offset in your diagonal should be `-(seq_len-1)` (where `seq_len` is the length of the random tokens which you repeat twice), because the second instance of random token `T` will attend to the token **after** the first instance of `T`.
</details>


<details><summary>Solution</summary>

```python
def induction_attn_detector(patterns: list[Tensor]) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be induction heads

    Remember - the tokens used to generate rep_cache are (bos_token, *rand_tokens, *rand_tokens)
    """
    attn_heads = []
    for layer in range(len(patterns)):
        for head in range(patterns[layer].shape[0]):
            attention_pattern = patterns[layer][head]
            # take avg of (-seq_len+1)-offset elements
            seq_len = (attention_pattern.shape[-1] - 1) // 2
            score = attention_pattern.diagonal(-seq_len + 1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```
</details>

<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

```python
# take avg of (-seq_len+1)-offset elements
seq_len = (attention_pattern.shape[-1] - 1) // 2
            score = attention_pattern.diagonal(-seq_len + 1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```

</details>


If this function works as expected, then you should see output that matches your observations from `circuitsvis` (i.e. the heads which you observed to be induction heads are being classified as induction heads by your function here).

# 3️⃣ Interventions with NNsight Traces

> ##### Learning Objectives
>
> - Understand how NNsight's trace context lets you read and modify activations
> - Use traces to access activations, process the results, and write them to an external tensor
> - Build tools to perform attribution, i.e. detecting which components of your model are responsible for performance on a given task
> - Understand how trace interventions can be used to perform basic interventions like **ablation**

# 3️⃣ NNsight: In-Trace Interventions & Logit Attribution

> ##### Learning Objectives
>
> - Understand how NNsight's trace context allows accessing and intervening on activations
> - Use `.save()` to extract activations and process results
> - Build tools to perform attribution, i.e. detecting which components of your model are responsible for performance on a given task
> - Understand how in-trace interventions can be used to perform basic interventions like **ablation**

## In-Trace Interventions with NNsight

One of the great things about interpreting neural networks is that we have *full control* over our system. We can make precise, surgical edits and see how the model's behaviour and other internals change. This is an extremely powerful tool for understanding model behaviour.

In NNsight, interventions happen **inside the trace context**. You can:

1. **Read activations**: Use `.output.save()` or `.input.save()` on any module
2. **Modify activations**: Assign new values to `.output` or `.input` within the trace
3. **Combine both**: Read a value, transform it, and write it back

```python
# Example: zero-ablating a specific head's output
with model.trace(tokens):
    # Read the current value
    attn_output = model.transformer.h[0].attn.output
    # Modify it (e.g., zero out head 3)
    attn_output[:, :, 3, :] = 0.0
    # Save the final logits to see the effect
    logits = model.lm_head.output.save()
```

<details>
<summary>Aside — comparison with TransformerLens hooks</summary>

In TransformerLens, you would use `model.run_with_hooks(tokens, fwd_hooks=[(hook_name, hook_fn)])` where `hook_fn` is a function that takes the activation tensor and a hook point object, and returns a modified tensor. NNsight's approach is more direct — you just modify the activation in-place within the trace context, without needing to define separate hook functions. Both approaches achieve the same thing, but NNsight's is often more readable for simple interventions.
</details>

## Induction Head Scoring

Let's use these techniques to compute induction scores for all heads. For our 2L model, we'll compute attention patterns directly and score each head.

In [19]:
if MAIN:
    seq_len = 50
    batch_size = 10
    rep_tokens_10 = generate_repeated_tokens(model, seq_len, batch_size)

    # Compute induction scores using nnterp's attention_probabilities accessor
    n_layers = model._model.n_layers
    n_heads = model._model.n_heads
    induction_score_store = t.zeros((n_layers, n_heads), device=device)

    # Get attention patterns via nnterp in a single trace
    patterns_10 = get_attention_patterns_2l(model, rep_tokens_10)

    for layer_idx in range(n_layers):
        pattern = patterns_10[layer_idx]  # [batch, n_heads, seq, seq]
        induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
        induction_score = einops.reduce(
            induction_stripe, "batch head_index position -> head_index", "mean"
        )
        induction_score_store[layer_idx, :] = induction_score

    imshow(
        induction_score_store,
        labels={"x": "Head", "y": "Layer"},
        title="Induction Score by Head",
        text_auto=".2f",
        width=900,
        height=350,
    )


### Interventions with NNsight Traces

Inside an `nnsight` trace context (`with model.trace(tokens):`), you can both **read** and **write** activations at any point in the model's computation graph.

To **read** an activation, call `.save()` on it — this marks it for extraction after the trace completes:
```python
with model.trace(tokens):
    attn_pattern = model.attention_probabilities[layer].save()
```

To **edit** an activation, simply assign to it inside the trace:
```python
with model.trace(tokens):
    model.attention_probabilities[layer][:, head] = 0.0  # zero-ablate a head
    logits = model.lm_head.output.save()
```

This is more direct than TransformerLens hooks — there's no separate hook function to define. You just read or modify activations inline within the trace context.

```python
def hook_function(
    attn_pattern: Float[Tensor, "batch heads seq_len seq_len"],
    hook: HookPoint
) -> Float[Tensor, "batch heads seq_len seq_len"]:

    # modify attn_pattern (can be inplace)
    return attn_pattern
```

### Running with hooks

Once you've defined a hook function (or functions), you should call `model.run_with_hooks`. A typical call to this function might look like:

```python
loss = model.run_with_hooks(
    tokens,
    return_type="loss",
    fwd_hooks=[
        ('blocks.1.attn.hook_pattern', hook_function)
    ]
)
```

### A bit more about hooks

<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

Here are a few extra notes for how to squeeze even more functionality out of hooks. If you'd prefer, you can [jump ahead](#hooks-accessing-activations) to see an actual example of hooks being used, and come back to this section later.

<details>
<summary>Resetting hooks</summary>

`model.run_with_hooks` has the default parameter `reset_hooks_end=True` which resets all hooks at the end of the run (including both those that were added before and during the run). Despite this, it's possible to shoot yourself in the foot with hooks, e.g. if there's an error in one of your hooks so the function never finishes. In this case, you can use `model.reset_hooks()` to reset all hooks.

If you don't want to reset hooks (i.e. you want to keep them between forward passes), you can either set `reset_hooks_end=False` in the `run_with_hooks` function, or just add the hooks directly using the `add_hook` method before your forward passes (this way they won't reset automatically).

</details>
<details>
<summary>Adding multiple hooks at once</summary>

Including more than one tuple in the `fwd_hooks` list is one way to add multiple hooks:

```python
loss = model.run_with_hooks(
    tokens,
    return_type="loss",
    fwd_hooks=[
        ('blocks.0.attn.hook_pattern', hook_function),
        ('blocks.1.attn.hook_pattern', hook_function)
    ]
)
```

Another way is to use a **name filter** rather than a single name:

```python
loss = model.run_with_hooks(
    tokens,
    return_type="loss",
    fwd_hooks=[
        (lambda name: name.endswith("pattern"), hook_function)
    ]
)
```
</details>
<details>
<summary><code>utils.get_act_name</code></summary>

When we were indexing the cache in the previous section, we found we could use strings like `cache['blocks.0.attn.hook_pattern']`, or use the shorthand of `cache['pattern', 0]`. The reason the second one works is that it calls the function `utils.get_act_name` under the hood, i.e. we have:

```python
utils.get_act_name('pattern', 0) == 'blocks.0.attn.hook_pattern'
```

Using `utils.get_act_name` in your forward hooks is often easier than using the full string, since the only thing you need to remember is the activation name (you can refer back to the diagram in the previous section for this).
</details>
<details>
<summary>Using <code>functools.partial</code> to create variations on hooks</summary>

A useful trick is to define a hook function with more arguments than it needs, and then use `functools.partial` to fill in the extra arguments. For instance, if you want a hook function which only modifies a particular head, but you want to run it on all heads separately (rather than just adding all the hooks and having them all run on the next forward pass), then you can do something like:

```python
def hook_all_attention_patterns(
    attn_pattern: Float[Tensor, "batch heads seq_len seq_len"],
    hook: HookPoint,
    head_idx: int
) -> Float[Tensor, "batch heads seq_len seq_len"]:
    # modify attn_pattern inplace, at head_idx
    return attn_pattern

for head_idx in range(12):
    temp_hook_fn = functools.partial(hook_all_attention_patterns, head_idx=head_idx)
    model.run_with_hooks(tokens, fwd_hooks=[('blocks.1.attn.hook_pattern', temp_hook_fn)])
```
</details>

And here are some points of interest, which aren't vital to understand:

<details>
<summary>Relationship to PyTorch hooks</summary>

[PyTorch hooks](https://blog.paperspace.com/pytorch-hooks-gradient-clipping-debugging/) are a great and underrated, yet incredibly janky, feature. They can act on a layer, and edit the input or output of that layer, or the gradient when applying autodiff. The key difference is that **Hook points** act on *activations* not layers. This means that you can intervene within a layer on each activation, and don't need to care about the precise layer structure of the transformer. And it's immediately clear exactly how the hook's effect is applied. This adjustment was shamelessly inspired by [Garcon's use of ProbePoints](https://transformer-circuits.pub/2021/garcon/index.html).

They also come with a range of other quality of life improvements. PyTorch's hooks are global state, which can be a massive pain if you accidentally leave a hook on a model. TransformerLens hooks are also global state, but `run_with_hooks` tries to create an abstraction where these are local state by removing all hooks at the end of the function (and they come with a helpful `model.reset_hooks()` method to remove all hooks).
</details>

<details>
<summary>How are TransformerLens hooks actually implemented?</summary>

They are implemented as modules with the identity function as their forward method:

```python
class HookPoint(nn.Module):
    ...
    def forward(self, x):
        return x
```

but also with special features for adding and removing hook functions. This is why you see hooks when you print a HookedTransformer model, because all its modules are recursively printed.

When you run the model normally, hook modules won't change the model's behaviour (since applying the identity function does nothing). It's only once you add functions to the hook modules (e.g. a function which ablates any inputs into the hook module) that the model's behaviour changes.

</details>

</details>

## Hooks: Accessing Activations

In later sections, we'll write some code to intervene on hooks, which is really the core feature that makes them so useful for interpretability. But for now, let's just look at how to access them without changing their value. This can be achieved by having the hook function write to a global variable, and return nothing (rather than modifying the activation in place).

Why might we want to do this? It turns out to be useful for things like:

* Extracting activations for a specific task
* Doing some long-running calculation across many inputs, e.g. finding the text that most activates a specific neuron

Note that, in theory, this could all be done using the `run_with_cache` function we used in the previous section, combined with post-processing of the cache result. But using hooks can be more intuitive and memory efficient.

### Exercise - calculate induction scores with traces

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 15-20 minutes on this exercise.
> This is our first exercise with trace-based interventions, a key NNsight technique. Use the hints if you're stuck.
> ```

To start with, we'll look at how trace contexts can be used to get the same results as from the previous section (where we computed induction head scores from saved attention patterns).

Most of the code has already been provided for you below; the only thing you need to do is **implement the `ablation_induction_score` function**. This function uses an NNsight trace to optionally zero-ablate a layer 0 head's attention pattern, then reads the resulting layer 1 attention pattern to compute an induction score.

Your function should do the following:

* Inside a `with model.trace(tokens):` block, optionally zero out the attention pattern for a given layer 0 head (by setting `model.attention_probabilities[0][:, head] = 0.0`).
* Save the layer 1 attention pattern using `model.attention_probabilities[1].save()`.
* After the trace, compute the induction score: the average of the diagonal offset by `-(seq_len_half - 1)` from the saved pattern for the specified layer 1 head.

<details>
<summary>Help - I'm not sure how to implement this function.</summary>

To get the induction stripe, you can use:

```python
torch.diagonal(pattern, dim1=-2, dim2=-1, offset=1-seq_len)
```

since this returns the diagonal of each attention scores matrix, for every element in the batch and every attention head.

Once you have this, you can then take the mean over the batch and diagonal dimensions, giving you a tensor of length `n_heads`. You can then write this to the global `induction_score_store` tensor at the correct layer index.
</details>

<details>
<summary>Solution</summary>

```python
# Get attention patterns via nnterp in a single trace
patterns_10 = get_attention_patterns_2l(model, rep_tokens_10)

for layer_idx in range(n_layers):
    pattern = patterns_10[layer_idx]  # [batch, n_heads, seq, seq]
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(
        induction_stripe, "batch head_index position -> head_index", "mean"
    )
    induction_score_store[layer_idx, :] = induction_score
```

</details>

If this function has been implemented correctly, you should see a result matching your observations from the previous section: a high induction score (>0.6) for all the heads which you identified as induction heads, and a low score (close to 0) for all others.

### Exercise - find induction heads in GPT2-small

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You shouldn't spend more than 10-20 minutes on this exercise.
> ```

Perform the same analysis on GPT-2 Small. You should observe that some heads, particularly in a couple of the middle layers, have high induction scores.

For GPT-2, we extract attention patterns from the `c_attn` outputs saved via NNsight's trace context.

In [20]:
# GPT-2 small induction head scoring
seq_len = 50
batch_size = 10
# Generate repeated tokens for GPT-2 (different vocab size)
t.manual_seed(0)
gpt2_bos = gpt2_small.tokenizer.bos_token_id
gpt2_prefix = (t.ones(batch_size, 1) * gpt2_bos).long()
gpt2_half = t.randint(0, 50257, (batch_size, seq_len), dtype=t.int64)
rep_tokens_gpt2 = t.cat([gpt2_prefix, gpt2_half, gpt2_half], dim=-1).to(device)

# For GPT-2, we need to extract attention patterns from c_attn outputs
gpt2_n_layers = 12
gpt2_n_heads = 12
gpt2_d_model = 768
gpt2_d_head = 64

induction_score_store_gpt2 = t.zeros((gpt2_n_layers, gpt2_n_heads), device=device)

# Run GPT-2 and collect all c_attn outputs
c_attn_outputs = []
with gpt2_small.trace(rep_tokens_gpt2):
    for layer_idx in range(gpt2_n_layers):
        c_attn_outputs.append(
            gpt2_small.layers[layer_idx].self_attn.c_attn.output.save()
        )

# Process each layer
for layer_idx in range(gpt2_n_layers):
    qkv = c_attn_outputs[layer_idx]  # [batch, seq, 3*d_model]
    q, k, v = qkv.split(gpt2_d_model, dim=-1)
    q = q.view(-1, rep_tokens_gpt2.shape[1], gpt2_n_heads, gpt2_d_head)
    k = k.view(-1, rep_tokens_gpt2.shape[1], gpt2_n_heads, gpt2_d_head)

    q_t = q.permute(0, 2, 1, 3)
    k_t = k.permute(0, 2, 1, 3)
    scores = (q_t @ k_t.transpose(-2, -1)) / gpt2_d_head**0.5
    seq_len_full = rep_tokens_gpt2.shape[1]
    causal_mask = t.triu(
        t.ones(seq_len_full, seq_len_full, dtype=t.bool, device=device), diagonal=1
    )
    scores.masked_fill_(causal_mask[None, None], float("-inf"))
    pattern = scores.softmax(-1)

    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(
        induction_stripe, "batch head_index position -> head_index", "mean"
    )
    induction_score_store_gpt2[layer_idx, :] = induction_score

imshow(
    induction_score_store_gpt2,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head (GPT-2 Small)",
    text_auto=".1f",
    width=700,
    height=500,
)


Perform the same analysis on your `gpt2_small`. You should observe that some heads, particularly in a couple of the middle layers, have high induction scores. Use CircuitsVis to plot the attention patterns for these heads when run on the repeated token sequences, and verify that they look like induction heads.

Note - you can make CircuitsVis plots (and other visualisations) using hooks rather than plotting directly from the cache. For example, we've given you a hook function which will display the attention patterns at a given hook when you include it in a call to `model.run_with_hooks`.

<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

```python
# Observation: heads 5.1, 5.5, 6.9, 7.2, 7.10 are all strongly induction-y.
# Confirm observation by visualizing attn patterns for layers 5 through 7:

induction_head_layers = [5, 6, 7]
fwd_hooks = [
    (utils.get_act_name("pattern", induction_head_layer), visualize_pattern_hook)
    for induction_head_layer in induction_head_layers
]
gpt2_small.run_with_hooks(
    rep_tokens,
    return_type=None,
    fwd_hooks=fwd_hooks,
)
```

</details>


## Building interpretability tools — Logit Attribution

In order to develop a mechanistic understanding for how transformers perform certain tasks, we need to be able to answer questions like:

> *How much of the model's performance on some particular task is attributable to each component of the model?*

where "component" here might mean, for example, a specific head in a layer.

There are many ways to approach a question like this. For example, we might look at how a head interacts with other heads in different layers, or we might perform a causal intervention by seeing how well the model performs if we remove the effect of this head. However, we'll keep things simple for now, and ask the question: **what are the direct contributions of this head to the output logits?**

### Direct Logit attribution

A consequence of the residual stream is that the output logits are the sum of the contributions of each layer, and thus the sum of the results of each head. This means we can decompose the output logits into a term coming from each head and directly do attribution like this!

<details>
<summary>A concrete example</summary>

Let's say that our model knows that the token Harry is followed by the token Potter, and we want to figure out how it does this. The logits on Harry are `residual @ W_U`. But this is a linear map, and the residual stream is the sum of all previous layers `residual = embed + attn_out_0 + attn_out_1`. So `logits = (embed @ W_U) + (attn_out_0 @ W_U) + (attn_out_1 @ W_U)`

We can be even more specific, and *just* look at the logit of the Potter token - this corresponds to a column of `W_U`, and so a direction in the residual stream - our logit is now a single number that is the sum of `(embed @ potter_U) + (attn_out_0 @ potter_U) + (attn_out_1 @ potter_U)`. Even better, we can decompose each attention layer output into the sum of the result of each head, and use this to get many terms.
</details>

Formally, for head $h$ at position $i$, the logit attribution for the correct next token $t_{i+1}$ is:

$$\text{logit\_attr}_{h,i} = (x_h^i)^T W_U[:, t_{i+1}]$$

where $x_h^i$ is the output of head $h$ at position $i$.

Your mission here is to write a function to look at how much each component contributes to the correct logit. Your components are:

* The direct path (i.e. the residual connections from the embedding to unembedding),
* Each layer 0 head (via the residual connection and skipping layer 1)
* Each layer 1 head

To emphasise, these are not paths from the start to the end of the model, these are paths from the output of some component directly to the logits - we make no assumptions about how each path was calculated!

A few important notes for this exercise:

* Here we are just looking at the DIRECT effect on the logits, i.e. the thing that this component writes / embeds into the residual stream - if heads compose with other heads and affect logits like that, or inhibit logits for other tokens to boost the correct one we will not pick up on this!
* By looking at just the logits corresponding to the correct token, our data is much lower dimensional because we can ignore all other tokens other than the correct next one (Dealing with a 50K vocab size is a pain!). But this comes at the cost of missing out on more subtle effects, like a head suppressing other plausible logits, to increase the log prob of the correct one.
    * There are other situations where our job might be easier. For instance, in the IOI task (which we'll discuss shortly) we're just comparing the logits of the indirect object to the logits of the direct object, meaning we can use the **difference between these logits**, and ignore all the other logits.
* When calculating correct output logits, we will get tensors with a dimension `(position - 1,)`, not `(position,)` - we remove the final element of the output (logits), and the first element of labels (tokens). This is because we're predicting the *next* token, and we don't know the token after the final token, so we ignore it.

<details>
<summary>Aside - centering <code>W_U</code></summary>

While we won't worry about this for this exercise, logit attribution is often more meaningful if we first center `W_U` - i.e. ensure the mean of each row writing to the output logits is zero. Log softmax is invariant when we add a constant to all the logits, so we want to control for a head that just increases all logits by the same amount. We won't do this here for ease of testing.
</details>

<details>
<summary>Question - why don't we do this to the log probs instead?</summary>

Because log probs aren't linear, they go through `log_softmax`, a non-linear function.
</details>

### Exercise - build logit attribution tool

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

Implement the `logit_attribution` function below. This should return the contribution of each component in the "correct direction".

In [21]:
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l1_results: Float[Tensor, "seq nheads d_model"],
    l2_results: Float[Tensor, "seq nheads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    """
    Inputs:
        embed: the embeddings of the tokens (i.e. token + position embeddings)
        l1_results: the outputs of the attention heads at layer 1 (with head as one of the dims)
        l2_results: the outputs of the attention heads at layer 2 (with head as one of the dims)
        W_U: the unembedding matrix
        tokens: the token ids of the sequence

    Returns:
        Tensor of shape (seq_len-1, n_components)
        represents the concatenation (along dim=-1) of logit attributions from:
            the direct path (seq-1,1)
            layer 0 logits (seq-1, n_heads)
            layer 1 logits (seq-1, n_heads)
        so n_components = 1 + 2*n_heads
    """
    W_U_correct_tokens = W_U[:, tokens[1:]]

    direct_attributions = einops.einsum(
        W_U_correct_tokens, embed[:-1], "emb seq, seq emb -> seq"
    )
    l1_attributions = einops.einsum(
        W_U_correct_tokens, l1_results[:-1], "emb seq, seq nhead emb -> seq nhead"
    )
    l2_attributions = einops.einsum(
        W_U_correct_tokens, l2_results[:-1], "emb seq, seq nhead emb -> seq nhead"
    )
    return t.concat(
        [direct_attributions.unsqueeze(-1), l1_attributions, l2_attributions], dim=-1
    )


def get_attn_head_results(
    model: StandardizedTransformer, tokens: Tensor
) -> tuple[Tensor, list[Tensor]]:
    """
    Run the 2L model and return per-head output contributions for each layer.
    Uses nnterp's attention_probabilities to get patterns, then decomposes per-head
    by computing V from weights and projecting through W_O.

    Returns:
        embed: [seq, d_model] — token + positional embeddings
        results: list of [seq, n_heads, d_model] tensors (one per layer)
    """
    raw = model._model
    n_layers = raw.n_layers

    saved_patterns = [None] * n_layers
    layer_outs = [None] * n_layers
    with model.trace(tokens):
        embed_out = model.embed.output.save()          # [batch, seq, d_model]
        pos_embed_out = model.pos_embed.output.save()   # [seq, d_model]
        for i in range(n_layers):
            saved_patterns[i] = model.attention_probabilities[i].save()
            layer_outs[i] = model.layers_output[i].save()

    embed = (embed_out.squeeze(0) + pos_embed_out)  # [seq, d_model]

    # Input to each layer's attention
    inputs = [
        embed_out + pos_embed_out,  # [batch, seq, d_model] for layer 0
        layer_outs[0],              # [batch, seq, d_model] for layer 1
    ]

    results = []
    for layer_idx in range(n_layers):
        attn = raw.blocks[layer_idx].attn
        inp = inputs[layer_idx]
        pattern = saved_patterns[layer_idx]  # [batch, n_heads, seq, seq]

        # V = input @ W_V + b_V
        v = t.einsum("bsd,hdi->bshi", inp, attn.W_V) + attn.b_V  # [batch, seq, n_heads, d_head]
        v = v.permute(0, 2, 1, 3)  # [batch, n_heads, seq, d_head]

        # z = pattern @ V per head
        z = pattern @ v  # [batch, n_heads, seq, d_head]
        z = z.permute(0, 2, 1, 3)  # [batch, seq, n_heads, d_head]

        # Per-head result: z_h @ W_O_h
        result = t.einsum("bsnh,nhm->bsnm", z, attn.W_O)  # [batch, seq, n_heads, d_model]
        results.append(result.squeeze(0))  # [seq, n_heads, d_model]

    return embed, results


In [22]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."
tokens = tokenizer_2l(text, return_tensors="pt").input_ids.to(device)

with model.trace(tokens):
    logits_out = model.lm_head.output.save()

embed, head_results = get_attn_head_results(model, tokens)
l1_results = head_results[0]
l2_results = head_results[1]

# W_U for the 2L model: unembed.weight is [d_vocab, d_model], we need [d_model, d_vocab]
W_U = model._model.unembed.weight.T  # [d_model, d_vocab]

with t.inference_mode():
    logit_attr = logit_attribution(embed, l1_results, l2_results, W_U, tokens[0])
    correct_token_logits = logits_out[0, t.arange(len(tokens[0]) - 1), tokens[0, 1:]]
    t.testing.assert_close(logit_attr.sum(1), correct_token_logits, atol=1e-3, rtol=0)
    print("Tests passed!")


Tests passed!


In [23]:
str_tokens = [tokenizer_2l.decode(t_) for t_ in tokens.squeeze()]
plot_logit_attribution(model, logit_attr, tokens, title="Logit attribution (demo prompt)")


<details><summary>Solution</summary>

```python
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l1_results: Float[Tensor, "seq nheads d_model"],
    l2_results: Float[Tensor, "seq nheads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    """
    Inputs:
        embed: the embeddings of the tokens (i.e. token + position embeddings)
        l1_results: the outputs of the attention heads at layer 1 (with head as one of the dims)
        l2_results: the outputs of the attention heads at layer 2 (with head as one of the dims)
        W_U: the unembedding matrix
        tokens: the token ids of the sequence

    Returns:
        Tensor of shape (seq_len-1, n_components)
        represents the concatenation (along dim=-1) of logit attributions from:
            the direct path (seq-1,1)
            layer 0 logits (seq-1, n_heads)
            layer 1 logits (seq-1, n_heads)
        so n_components = 1 + 2*n_heads
    """
    W_U_correct_tokens = W_U[:, tokens[1:]]

    direct_attributions = einops.einsum(W_U_correct_tokens, embed[:-1], "emb seq, seq emb -> seq")
    l1_attributions = einops.einsum(
        W_U_correct_tokens, l1_results[:-1], "emb seq, seq nhead emb -> seq nhead"
    )
    l2_attributions = einops.einsum(
        W_U_correct_tokens, l2_results[:-1], "emb seq, seq nhead emb -> seq nhead"
    )
    return t.concat([direct_attributions.unsqueeze(-1), l1_attributions, l2_attributions], dim=-1)
```
</details>

Once you've got the tests working, you can visualise the logit attributions for each path through the model. We've provided you with the helper function `plot_logit_attribution`, which presents the results in a nice way.

#### Question - what is the interpretation of this plot?

You should find that the most variation in the logit attribution comes from the direct path. In particular, some of the tokens in the direct path have a very high logit attribution (e.g. tokens that begin common bigrams like `manip` -> `ulative`).

<details>
<summary>Answer - what is special about these tokens?</summary>

The tokens with very high logit attribution are the ones which are the first token in common bigrams. For instance, the highest contribution on the direct path comes from `| manip|`, because this is very likely to be followed by `|ulative|`. These predictions come from the direct path (embeddings -> unembedding) rather than from any attention head, because the model has learned these bigram statistics directly in the embedding/unembedding matrices.
</details>

### Exercise - interpret logit attribution for the induction heads

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

*This exercise just involves calling `logit_attribution` and `plot_logit_attribution` with appropriate arguments - the important part is interpreting the results. Please do look at the solutions if you're stuck on the code; this part isn't important.*

Perform logit attribution for your attention-only model `model`, on the cached activations from the repeated token sequence. What do you expect to see?

In [24]:
seq_len = 50
rep_tokens_1 = generate_repeated_tokens(model, seq_len, batch_size=1)

embed_rep, head_results_rep = get_attn_head_results(model, rep_tokens_1)
l1_results_rep = head_results_rep[0]
l2_results_rep = head_results_rep[1]

logit_attr_rep = logit_attribution(
    embed_rep, l1_results_rep, l2_results_rep, W_U, rep_tokens_1.squeeze()
)
plot_logit_attribution(
    model,
    logit_attr_rep,
    rep_tokens_1.squeeze(),
    title="Logit attribution (random induction prompt)",
)


<details>
<summary>Interpretation</summary>

The first half of the plot is mostly meaningless, because the sequences here are random and carry no predictable pattern.

In the second half, we see that heads `1.4` and `1.10` have a large logit attribution score. This makes sense given our previous observation that these heads seemed to be performing induction. This plot gives us a different kind of evidence than looking at attention patterns — it tells us not just *what* the heads attend to, but that they are actually *contributing to the correct logit* in the output.
</details>

<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

<details><summary>Solution</summary>

```python
seq_len = 50

embed = rep_cache["embed"]
l1_results = rep_cache["result", 0]
l2_results = rep_cache["result", 1]

logit_attr = logit_attribution(embed, l1_results, l2_results, model.W_U, rep_tokens.squeeze())
plot_logit_attribution(
    model, logit_attr, rep_tokens.squeeze(), title="Logit attribution (random induction prompt)"
)
```
</details>

</details>

## Ablation Studies

Now that we've built some tools to decompose our model's output, it's time to start making causal interventions.

### Ablations

An ablation is a simple causal intervention on a model - we pick some part of it and set it to zero. This is a crude proxy for how much that part matters. Further, if we have some story about how a specific circuit in the model enables some capability, showing that ablating *other* parts does nothing can be strong evidence of this.

As mentioned in [the glossary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=fh-HJyz1CgUVrXuoiban6bYx), there are many ways to do ablation. We'll focus on zero-ablation and mean-ablation.

### Exercise - induction head ablation

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should aim to spend 20-35 mins on this exercise.
> ```

Implement a function that ablates individual attention heads and measures the effect on cross-entropy loss. For each head, we zero out its contribution to the residual stream during the forward pass, then measure how much the loss increases.

You need to implement `get_ablation_scores` which returns a tensor of shape `(n_layers, n_heads)` containing the increase in loss from ablating each head. The approach is:

1. Run the model without any ablation to get a baseline loss.
2. For each head in each layer, run the model while zeroing out that head's output (the vectors we get when taking a weighted sum of the value vectors according to the attention probabilities, before projecting them up & adding them back to the residual stream).
3. Record the difference in loss.

A few tips:

- Use NNsight's trace context to intervene on the head outputs. Within the trace, you can set a specific head's output to zero by indexing into the appropriate activation tensor and assigning zeros.
- We only care about the loss on the second half of the sequence (the repeated tokens). The sequences have length `2 * seq_len + 1` (a BOS token plus 2 repeated random sequences), so only take the last `seq_len - 1` tokens when computing loss.
- If you're confused about what different activations mean, you can refer back to [the diagram](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/small-merm.svg).

<details>
<summary>🛑 <span style="color:red;font-weight:bold">TransformerLens equivalent</span></summary>

In TransformerLens, this would be done using hooks:

```python
def head_zero_ablation_hook(z, hook, head_index_to_ablate):
    z[:, :, head_index_to_ablate, :] = 0.0
    return z

model.run_with_hooks(
    tokens,
    fwd_hooks=[(utils.get_act_name("z", layer), partial(head_zero_ablation_hook, head_index_to_ablate=head))]
)
```

In NNsight, the equivalent is done inside a `model.trace()` context by directly assigning to the activation tensor.
</details>

In [25]:
def get_ablation_scores(
    model: StandardizedTransformer,
    tokens: Int[Tensor, "batch seq"],
    ablation_type: str = "zero",
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns a tensor of shape (n_layers, n_heads) containing the increase in cross entropy loss
    from ablating the output of each head.

    For zero ablation: zeroes the attention pattern for the target head inside an nnterp trace.
    For mean ablation: computes a correction to replace head output with batch-mean output.
    """
    raw = model._model
    n_layers, n_heads = raw.n_layers, raw.n_heads
    ablation_scores = t.zeros((n_layers, n_heads), device=device)
    seq_len = (tokens.shape[1] - 1) // 2

    # Baseline loss
    with model.trace(tokens):
        logits_clean = model.lm_head.output.save()
    loss_no_ablation = -get_log_probs(logits_clean, tokens)[:, -(seq_len - 1) :].mean()

    # Pre-compute for mean ablation: patterns, layer inputs → per-head z corrections
    if ablation_type == "mean":
        all_patterns = [None] * n_layers
        layer_outs = [None] * n_layers
        with model.trace(tokens):
            embed_out = model.embed.output.save()
            pos_embed_out = model.pos_embed.output.save()
            for i in range(n_layers):
                all_patterns[i] = model.attention_probabilities[i].save()
                layer_outs[i] = model.layers_output[i].save()
        layer_inputs = [embed_out + pos_embed_out, layer_outs[0]]

    for layer in tqdm(range(n_layers)):
        # Pre-compute V and z for all heads in this layer (mean ablation only)
        if ablation_type == "mean":
            attn = raw.blocks[layer].attn
            v = t.einsum("bsd,hdi->bshi", layer_inputs[layer], attn.W_V) + attn.b_V
            v = v.permute(0, 2, 1, 3)  # [batch, n_heads, seq, d_head]
            z_all = all_patterns[layer] @ v  # [batch, n_heads, seq, d_head]

        for head in range(n_heads):
            with model.trace(tokens):
                if ablation_type == "zero":
                    model.attention_probabilities[layer][:, head] = 0.0
                else:
                    z_h = z_all[:, head]  # [batch, seq, d_head]
                    mean_z_h = z_h.mean(dim=0, keepdim=True).expand_as(z_h)
                    correction = (mean_z_h - z_h) @ attn.W_O[head]  # [batch, seq, d_model]
                    model.attentions_output[layer] = model.attentions_output[layer] + correction
                logits = model.lm_head.output.save()
            loss = -get_log_probs(logits, tokens)[:, -(seq_len - 1) :].mean()
            ablation_scores[layer, head] = loss - loss_no_ablation

    return ablation_scores


In [26]:
rep_tokens_1 = generate_repeated_tokens(model, seq_len=50, batch_size=1)
ablation_scores = get_ablation_scores(model, rep_tokens_1)


/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:288: UserWarning:

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)

/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:292: UserWarning:

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)

/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:296: UserWa

In [27]:
imshow(
    ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads",
    text_auto=".2f",
    width=900,
    height=350,
)


<details>
<summary>Interpretation</summary>

This tells us not just which heads are responsible for writing output that gets us the correct result, but **which heads play an important role in the induction circuit**.

Head `0.7` is by far the most important in layer 0 (which makes sense, since we observed it to be the strongest "previous token head"), and heads `1.4`, `1.10` are the most important in layer 1 (which makes sense, since we observed these to be the most induction-y).
</details>

<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

```python
# Initialize an object to store the ablation scores
ablation_scores = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)

    # Calculating loss without any ablation, to act as a baseline
    model.reset_hooks()
    seq_len = (tokens.shape[1] - 1) // 2
    logits = model(tokens, return_type="logits")
    loss_no_ablation = -get_log_probs(logits, tokens)[:, -(seq_len - 1) :].mean()

    for layer in tqdm(range(model.cfg.n_layers)):
        for head in range(model.cfg.n_heads):
            # Use functools.partial to create a temporary hook function with the head number fixed
            temp_hook_fn = functools.partial(ablation_function, head_index_to_ablate=head)
            # Run the model with the ablation hook
            ablated_logits = model.run_with_hooks(
                tokens, fwd_hooks=[(utils.get_act_name("z", layer), temp_hook_fn)]
            )
            # Calculate the loss difference (= neg correct logprobs), only on the last seq_len tokens
            loss = -get_log_probs(ablated_logits, tokens)[:, -(seq_len - 1) :].mean()
            # Store the result, subtracting the clean loss so that a value of 0 means no loss change
            ablation_scores[layer, head] = loss - loss_no_ablation

    return ablation_scores
```

</details>


Once you've passed the tests, you can plot the results:

### Exercise - mean ablation

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should aim to spend 5-15 mins on this exercise.
> ```

An alternative to zero-ablation is **mean-ablation**, where rather than setting values to zero, we set them to be their mean across some suitable distribution (commonly we'll use the mean over some batch dimension). This can be more informative, because zero-ablation takes a model out of its normal distribution, and so the results from it aren't necessarily representative of what you'd get if you "switched off" the effect from some particular component. Mean ablation on the other hand works slightly better (although it does come with its own set of risks). You can read more [here](https://www.neelnanda.io/mechanistic-interpretability/glossary#:~:text=Ablation%20aka%20Knockout) or [here](https://arxiv.org/html/2404.15255v1).

You should modify your `get_ablation_scores` function to support `ablation_type="mean"` and run the code below. You should see that the results are slightly cleaner, with the unimportant heads having values much closer to zero relative to the important heads.

In [28]:
rep_tokens_batch = generate_repeated_tokens(model, seq_len=50, batch_size=10)
mean_ablation_scores = get_ablation_scores(
    model, rep_tokens_batch, ablation_type="mean"
)

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads (Mean)",
    text_auto=".2f",
    width=900,
    height=350,
)


/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:288: UserWarning:

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)

/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:292: UserWarning:

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)

/home/dev/arena-venv/lib/python3.12/site-packages/eindex/indexing.py:296: UserWa

<details><summary>Solution</summary>

```python
rep_tokens_batch = generate_repeated_tokens(model, seq_len=50, batch_size=10)
mean_ablation_scores = get_ablation_scores(
    model, rep_tokens_batch, ablation_type="mean"
)

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads (Mean)",
    text_auto=".2f",
    width=900,
    height=350,
)
```
</details>

<details>
<summary>🛑 <span style="color:red;font-weight:bold">Original (TransformerLens)</span></summary>

```python
## Bonus - understand heads 0.4 & 0.11 (very hard!)
There are 2 heads which appeared strongly in our induction ablation experiments, but haven't stood out as much in the other analysis we've done in this section `0.4` and `0.11`. Can you construct causal experiments (i.e. targeted ablations) to try and figure out what these heads are doing?

> Note - you might want to attempt this once you've made some headway into the next section, as this will give you a more mechanistic understanding of the induction circuit. Even once you've done that, you might still find this bonus exercise challenging, because it ventures outside of the well-defined induction circuit we've been working with and into potentially more ambiguous results. **To restate - the material here is very challenging!**

<details>
<summary>Here's a hint to get you started</summary>

Look at the positions that heads `0.4` and `0.11` are attending to. Can you figure out which source positions are important to attend to for the model to perform well?

</details>

<details>
<summary>Partial answer (and some sample code)</summary>

Below is some sample code which plots the effect of ablating the inputs to heads `0.4` and `0.11` at all offset positions minus a few (e.g. the first row shows the effect on loss of mean ablating all inputs to the heads except for those that come from self-attention, and the second row shows the effect when we ablate all inputs except for those that come from the token immediately before it in the sequence). 

```python
def head_z_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
    seq_posns: list[int],
    cache: ActivationCache,
) -> None:
    """
    We perform ablation at the z vector, by doing the equivalent of mean ablating all the inputs to this attention head
    except for those which come from the tokens `n` positions back, where `n` is in the `seq_posns` list.
    """
    batch, seq = z.shape[:2]
    v = cache["v", hook.layer()][:, :, head_index_to_ablate]  # shape [batch seq_K d_head]
    pattern = cache["pattern", hook.layer()][:, head_index_to_ablate]  # shape [batch seq_Q seq_K]

    # Get a repeated version of v, and mean ablate all but the previous token values
    v_repeated = einops.repeat(v, "b sK h -> b sQ sK h", sQ=seq)
    v_ablated = einops.repeat(v_repeated.mean(0), "sQ sK h -> b sQ sK h", b=batch).clone()
    for offset in seq_posns:
        seqQ_slice = t.arange(offset, seq)
        v_ablated[:, seqQ_slice, seqQ_slice - offset] = v_repeated[:, seqQ_slice, seqQ_slice - offset]

    # Take weighted sum of this new v, and use it to edit `z` inplace.
    z[:, :, head_index_to_ablate] = einops.einsum(v_ablated, pattern, "b sQ sK h, b sQ sK -> b sQ h")


def get_ablation_scores_cache_assisted(
    model: HookedTransformer,
    tokens: Int[Tensor, "batch seq"],
    ablation_function: Callable = head_zero_ablation_hook,
    seq_posns: list[int] = [0],
    layers: list[int] = [0],
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Version of `get_ablation_scores` which can use the cache to assist with the ablation.
    """
    ablation_scores = t.zeros((len(layers), model.cfg.n_heads), device=model.cfg.device)

    model.reset_hooks()
    seq_len = (tokens.shape[1] - 1) // 2
    logits, cache = model.run_with_cache(tokens, return_type="logits")
    loss_no_ablation = -get_log_probs(logits, tokens)[:, -(seq_len - 1) :].mean()

    for layer in layers:
        for head in range(model.cfg.n_heads):
            temp_hook_fn = functools.partial(ablation_function, head_index_to_ablate=head, cache=cache, seq_posns=seq_posns)
            ablated_logits = model.run_with_hooks(tokens, fwd_hooks=[(utils.get_act_name("z", layer), temp_hook_fn)])
            loss = -get_log_probs(ablated_logits, tokens)[:, -(seq_len - 1) :].mean()
            ablation_scores[layer, head] = loss - loss_no_ablation

    return ablation_scores


rep_tokens_batch = run_and_cache_model_repeated_tokens(model, seq_len=50, batch_size=50)[0]

offsets = [[0], [1], [2], [3], [1, 2], [1, 2, 3]]
z_ablation_scores = [
    get_ablation_scores_cache_assisted(model, rep_tokens_batch, head_z_ablation_hook, offset).squeeze()
    for offset in tqdm(offsets)
]

imshow(
    t.stack(z_ablation_scores),
    labels={"x": "Head", "y": "Position offset", "color": "Logit diff"},
    title="Loss Difference (ablating heads everywhere except for certain offset positions)",
    text_auto=".2f",
    y=[str(offset) for offset in offsets],
    width=900,
    height=400,
)
```


Some observations from the result of this code:

- **Head `0.7` is truly a previous token head.** The second row shows that mean ablating all its inputs except for those that come from the previous token has no effect on loss, so this is all the information it's using.
- **Head `0.11` is only a current token head.** The first row shows that mean ablating all its inputs except for those that come from self-attending (i.e. to the current token) has no effect on loss, so this is all the information it's using.
- **Head `0.4` is only using information from positions 1, 2 or 3 tokens back.** This is shown from the 5th row of the plot above - the effect of ablating all inputs except for those that come from tokens 1 or 2 positions back is very small. Note that it's important we draw this conclusion from an ablation experiment, not just from looking at attention patterns - because attending to a token doesn't tell you whether that token is being used for a way that's important in the context of this particular distribution (induction).

Starting with `0.11` - we know that there are heads in layer 1 whose job it is to copy tokens - i.e. in sequences `[A][B]...[A][B]`, they attend from the second `[A]` back to the first `[B]` and copy its value to use as a prediction. And if head `0.11` always self-attends, then it actually makes sense to consider `(embedding of B) + (output of head 0.11 when it attends to token B)` as the "true embedding of `B`", since this is always the thing that the layer 1 head will be learning to copy. This idea of an **extended embedding** or **effective embedding** will come up again later in the course, when we look at GPT2-Small. As for whether the output of `0.11` is more important in the QK circuit of the layer-1 copying head, or the OV copying head, we'll leave as an exercise to the reader!

Next, `0.4` - it's using information from both 1 and 2 tokens back. Using the previous token makes sense, since induction circuits contain previous token heads. But what could it be doing with the information 2 positions back? One theory we might have is that it's also creating an induction circuit, but using 3 tokens rather than 2 tokens! In other words, rather than having sequences like `[A][B]...[A][B]` where the second `[A]` attends back to "token that came immediately after the value of this token", we might have sequences like `[Z][A][B]...[Z][A][B]` where the second `[A]` attends back to "token that came 2 positions after the value of the previous token". One way to test this would be to construct random induction sequences which have a maximum of 2 repetitions, i.e. they're constructed with the first half being random sequences and the second half being pairs of randomly chosen tokens which appear in the first half adjacent to each other. To illustrate, for vocab size of 10 and half seq len of 10, we might have a sequence like:

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">0 5 8 3 1 8 2 2 4 6 (5 8) (4 6) (3 1) (2 4) (0 5)</pre>

Based on our theory about head `0.4`, we should expect that mean ablating it in this kind of sequence should have nearly zero effect on loss (because it's designed to support induction sequences of length at least 3), even though all the other heads which were identified as important in the induction experiment (`0.7`, `0.11`, `1.4`, `1.10`) should still be important. This is in fact what we find - you can try this for yourself with the code below.

```python
def generate_repeated_tokens_maxrep(
    model: HookedTransformer,
    seq_len: int,
    batch_size: int = 1,
    maxrep: int = 2,
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Same as previous function, but contains a max number of allowed repetitions. For example, maxrep=2 means we can have
    sequences like `[A][B]...[A][B]`, but not `[A][B][C]...[A][B][C]`.
    """
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()
    rep_tokens_half = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.int64)
    rep_tokens = t.cat([prefix, rep_tokens_half], dim=-1)
    for _ in range(seq_len // maxrep + 1):
        random_start_posn = t.randint(0, seq_len - 2, (batch_size,)).tolist()
        rep_tokens_repeated = t.stack([rep_tokens_half[b, s : s + maxrep] for b, s in enumerate(random_start_posn)])
        rep_tokens = t.cat([rep_tokens, rep_tokens_repeated], dim=-1)

    return rep_tokens[:, : 2 * seq_len + 1].to(device)


rep_tokens_max2 = generate_repeated_tokens_maxrep(model, seq_len=50, batch_size=50, maxrep=2)

mean_ablation_scores = get_ablation_scores(model, rep_tokens_max2, ablation_fn=head_mean_ablation_hook)

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads",
    text_auto=".2f",
    width=900,
    height=350,
)
```

</details></details>


## Refresher - the induction circuit

Before we get into the meat of this section, let's refresh the results we've gotten so far from investigating induction heads. We've found:

* When fed repeated sequences of tokens, heads `1.4` and `1.10` have the characteristic induction head attention pattern of a diagonal stripe with offset `seq_len - 1`.
    * We saw this both from the CircuitsVis results, and from the fact that these heads had high induction scores by our chosen metric (with all other heads having much lower scores).
* We also saw that head `0.7` strongly attends to the previous token in the sequence (even on non-repeated sequences).
* We performed **logit attribution** on the model, and found that the values written to the residual stream by heads `1.4` and `1.10` were both important for getting us correct predictions in the second half of the sequence.
* We performed **zero-ablation** on the model, and found that heads `0.7`, `1.4` and `1.10` all resulted in a large accuracy degradation on the repeated sequence task when they were ablated.

Based on all these observations, try and summarise the induction circuit and how it works, in your own words. You should try and link your explanation to the QK and OV circuits for particular heads, and describe what type (or types) of attention head composition are taking place.

You can use the dropdown below to check your understanding.

<details>
<summary>My summary of the algorithm</summary>

* Head `0.7` is a previous token head (the QK-circuit ensures it always attends to the previous token).
* The OV circuit of head `0.7` writes a copy of the previous token in a *different* subspace to the one used by the embedding.
* The output of head `0.7` is used by the *key* input of head `1.10` via K-Composition to attend to 'the source token whose previous token is the destination token'.
* The OV-circuit of head `1.10` copies the *value* of the source token to the same output logit.
    * Note that this is copying from the embedding subspace, *not* the `0.7` output subspace - it is not using V-Composition at all.
* `1.4` is also performing the same role as `1.10` (so together they can be more accurate - we'll see exactly how later).

To emphasise - the sophisticated hard part is computing the *attention* pattern of the induction head - this takes careful composition. The previous token and copying parts are fairly easy. This is a good illustrative example of how the QK circuits and OV circuits act semi-independently, and are often best thought of somewhat separately. And that computing the attention patterns can involve real and sophisticated computation!

Below is a diagram of the induction circuit, with the heads indicated in the weight matrices.

![kcomp_diagram_3.png](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described_3.png)
</details>

# 4️⃣ Reverse-engineering induction circuits

> ##### Learning Objectives
>
> - Understand the difference between investigating a circuit by looking at activation patterns, and reverse-engineering a circuit by looking directly at the weights
> - Use explicit weight matrix multiplication to inspect the QK and OV circuits within an induction circuit
> - Perform further exploration of induction circuits: composition scores, and targeted ablations

In previous exercises, we looked at the attention patterns and attributions of attention heads to try and identify which ones were important in the induction circuit. This might be a good way to get a feel for the circuit, but it's not a very rigorous way to understand it. It would be better described as **feature analysis**.

Now we're going to do some more rigorous mechanistic analysis - digging into the weights and using them to reverse engineer the induction circuit.

## Refresher - the induction circuit

We've found:

* When fed repeated sequences of tokens, heads `1.4` and `1.10` have the characteristic induction head attention pattern of a diagonal stripe with offset `seq_len - 1`.
* Head `0.7` strongly attends to the previous token (a "previous token head").
* Ablating head `0.7` significantly reduces the induction scores of heads `1.4` and `1.10`.

The induction circuit hypothesis is:
1. Head `0.7` copies information about the previous token into each position's residual stream.
2. Heads `1.4` and `1.10` use this information (via K-composition) to identify positions where the current token appeared before, then copy the next token (via their OV circuit).

## Refresher - QK and OV circuits

As a shorthand, I'll often refer to $W_V^h W_O^h$ as the **OV circuit** of head $h$, and $W_Q^h (W_K^h)^T$ as the **QK circuit**.

* The **OV circuit** $W_{OV}^h = W_V^h W_O^h$ has shape `[d_model, d_model]`. It describes **what information gets moved** from source to destination position.
* The **QK circuit** $W_{QK}^h = W_Q^h (W_K^h)^T$ has shape `[d_model, d_model]`. It describes **where information is moved from** (i.e., which source positions the head attends to).

The full OV circuit, going from input tokens to output logits, is:

$$W_E \cdot W_{OV}^h \cdot W_U$$

This is a `[d_vocab, d_vocab]` matrix that tells us: "if the head attends to a token $t$ at the source position, what is the resulting contribution to the output logits?"

<details>
<summary>Aside — FactoredMatrix class in TransformerLens</summary>

TransformerLens provides a `FactoredMatrix` class that lets you work with large matrices (like the `[d_vocab, d_vocab]` full circuit matrix) without ever materializing them in memory. It stores them as a product of two smaller matrices and provides methods for computing properties like eigenvalues, singular values, and norms efficiently.

In our NNsight approach, we use explicit matrix multiplication with `torch.einsum` and standard tensor operations. This is more transparent and educational — you see exactly what computation is being done. For very large matrices, you can still avoid materializing the full product by working with subsets of rows/columns.
</details>

#### Question - what is the interpretation of each of the following matrices?

*There are quite a lot of questions here, but they are conceptually important. If you're confused, you might want to read the answers to the first few questions and then try the later ones.*

In your answers, you should describe the type of input it takes, and what the outputs represent.

#### $W_{OV}^{h}$

<details>
<summary>Answer</summary>

$W_{OV}^{h}$ has size $(d_\text{model}, d_\text{model})$, it is a linear map describing **what information gets moved from source to destination, in the residual stream.**

In other words, if $x$ is a vector in the residual stream, then $x^T W_{OV}^{h}$ is the vector written to the residual stream at the destination position, if the destination token only pays attention to the source token at the position of the vector $x$.
</details>

#### $W_E W_{OV}^h W_U$

<details>
<summary>Hint</summary>

If $A$ is the one-hot encoding for token `A` (i.e. the vector with zeros everywhere except for a one in the position corresponding to token `A`), then think about what $A^T W_E W_{OV}^h W_U$ represents. You can evaluate this expression from left to right (e.g. start with thinking about what $A^T W_E$ represents, then multiply by the other two matrices).
</details>
<details>
<summary>Answer</summary>

$W_E W_{OV}^h W_U$ has size $(d_\text{vocab}, d_\text{vocab})$, it is a linear map describing **what information gets moved from source to destination, in a start-to-end sense.**

If $A$ is the one-hot encoding for token `A`, then:

* $A^T W_E$ is the embedding vector for `A`.
* $A^T W_E W_{OV}^h$ is the vector which would get written to the residual stream at the destination position, if the destination token only pays attention to `A`.
* $A^T W_E W_{OV}^h W_U$ is the unembedding of this vector, i.e. the thing which gets added to the final logits.

</details>

#### $W_{QK}^{h}$

<details>
<summary>Answer</summary>

$W_{QK}^{h}$ has size $(d_\text{model}, d_\text{model})$, it is a bilinear form describing **where information is moved to and from** in the residual stream (i.e. which residual stream vectors attend to which others).

$x_i^T W_{QK}^h x_j = (x_i^T W_Q^h) (x_j^T W_K^h)^T$ is the attention score paid by token $i$ to token $j$.
</details>

#### $W_E W_{QK}^h W_E^T$

<details>
<summary>Answer</summary>

$W_E W_{QK}^h W_E^T$ has size $(d_\text{vocab}, d_\text{vocab})$, it is a bilinear form describing **where information is moved to and from**, among words in our vocabulary (i.e. which tokens pay attention to which others).

If $A$ and $B$ are one-hot encodings for tokens `A` and `B`, then $A^T W_E W_{QK}^h W_E^T B$ is the attention score paid by token `A` to token `B`:

$$
A^T \, W_E\, W_{QK}^{h}\, W_E^T \, B = \underbrace{(A^T W_E W_Q^{h})}_{\text{query for token } A}  \underbrace{(B^T W_E W_K^{h})^T}_{\text{key for token }B}
$$
</details>

#### $W_{pos} W_{QK}^h W_{pos}^T$

<details>
<summary>Answer</summary>

$W_{pos} W_{QK}^h W_{pos}^T$ has size $(n_\text{ctx}, n_\text{ctx})$, it is a bilinear form describing **where information is moved to and from**, among tokens in our context (i.e. which token positions pay attention to other positions).

If $i$ and $j$ are one-hot encodings for positions `i` and `j` (in other words they are just the ith and jth basis vectors), then $i^T W_{pos} W_{QK}^h W_{pos}^T j$ is the attention score paid by the token with position `i` to the token with position `j`:

$$
i^T \, W_{pos}\, W_{QK}^{h}\, W_{pos}^T \, j = \underbrace{(i^T W_{pos} W_Q^{h})}_{\text{query for i-th token}}  \underbrace{(j^T W_{pos} W_K^{h})^T}_{\text{key for j-th token}}
$$

</details>

#### $W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T$

where $h_1$ is in an earlier layer than $h_2$.

<details>
<summary>Hint</summary>

This matrix is best seen as a bilinear form of size $(d_\text{vocab}, d_\text{vocab})$. The $(A, B)$-th element is:

$$
(A^T W_E W_{OV}^{h_1}) W_{QK}^{h_2} (B^T W_E)^T
$$
</details>

<details>
<summary>Answer</summary>

$W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T$ has size $(d_\text{vocab}, d_\text{vocab})$, it is a bilinear form describing where information is moved to and from in head $h_2$, given that the **query-side vector** is formed from the output of head $h_1$. In other words, this is an instance of **Q-composition**.

If $A$ and $B$ are one-hot encodings for tokens `A` and `B`, then $A^T W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T B$ is the attention score paid **to** token `B`, **by** any token which attended strongly to an `A`-token in head $h_1$.

---

To further break this down, if it still seems confusing:

$$
\begin{aligned}
A^T \, W_E\, W_{OV}^{h_1} W_{QK}^{h_2}\, W_E^T \, B &= \underbrace{(A^T W_E W_{OV}^{h_1}W_Q^{h_2})}_{\text{query of token which attended to A}}  \underbrace{(B^T W_E W_K^{h_2})^T}_\text{key of token B} \\
\end{aligned}
$$

---

Note that the actual attention score will be a sum of multiple terms, not just this one (in fact, we'd have a different term for every combination of query and key input). But this term describes the **particular contribution** to the attention score from this combination of query and key input, and it might be the case that this term is the only one that matters (i.e. all other terms don't much affect the final probabilities). We'll see something exactly like this later on.
</details>

Before we start, there's a problem that we might run into when calculating all these matrices. Some of them are massive, and might not fit on our GPU. For instance, both full circuit matrices have shape $(d_\text{vocab}, d_\text{vocab})$, which in our case means $50278\times 50278 \approx 2.5\times 10^{9}$ elements. Even if your GPU can handle this, it still seems inefficient. Is there any way we can meaningfully analyse these matrices, without actually having to calculate them?

## Factored Matrix class

In transformer interpretability, we often need to analyse low rank factorized matrices - a matrix $M = AB$, where M is `[large, large]`, but A is `[large, small]` and B is `[small, large]`. This is a common structure in transformers.

For instance, we can factorise the OV circuit above as $W_{OV}^h = W_V^h W_O^h$, where $W_V^h$ has shape `[768, 64]` and $W_O^h$ has shape `[64, 768]`. For an even more extreme example, the full OV circuit can be written as $(W_E W_V^h) (W_O^h W_U)$, where these two matrices have shape `[50278, 64]` and `[64, 50278]` respectively. Similarly, we can write the full QK circuit as $(W_E W_Q^h) (W_E W_K^h)^T$.

The `FactoredMatrix` class is a convenient way to work with these. It implements efficient algorithms for various operations on these, such as computing the trace, eigenvalues, Frobenius norm, singular value decomposition, and products with other matrices. It can (approximately) act as a drop-in replacement for the original matrix.

This is all possible because knowing the factorisation of a matrix gives us a much easier way of computing its important properties. Intuitively, since $M=AB$ is a very large matrix that operates on very small subspaces, we shouldn't expect knowing the actual values $M_{ij}$ to be the most efficient way of storing it!

### Exercise - deriving properties of a factored matrix

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 10-25 minutes on this exercise.
> 
> If you're less interested in the maths, you can skip these exercises.
> ```

To give you an idea of what kinds of properties you can easily compute if you have a factored matrix, let's try and derive some ourselves.

Suppose we have $M=AB$, where $A$ has shape $(m, n)$, $B$ has shape $(n, m)$, and $m > n$. So $M$ is a size-$(m, m)$ matrix with rank at most $n$.

**Question - how can you easily compute the trace of $M$?**

<details>
<summary>Answer</summary>

We have:

$$
\text{Tr}(M) = \text{Tr}(AB)
= \sum_{i=1}^m \sum_{j=1}^n A_{ij} B_{ji}
$$

so evaluation of the trace is $O(mn)$.

Note that, by cyclicity of the trace, we can also show that $\text{Tr}(M) = \text{Tr}(BA)$ (although we don't even need to calculate the product $AB$ to evaluate the trace).
</details>

**Question - how can you easily compute the eigenvalues of $M$?**

(As you'll see in later exercises, eigenvalues are very important for evaluating matrices, for instance we can assess the [copying scores](https://transformer-circuits.pub/2021/framework/index.html#copying-matrix) of an OV circuit by looking at the eigenvalues of $W_{OV}$.)

<details>
<summary>Hint</summary>

It's computationally cheaper to find the eigenvalues of $BA$ rather than $AB$.

How are the eigenvalues of $AB$ and $BA$ related?
</details>
<details>
<summary>Answer</summary>

The eigenvalues of $AB$ and $BA$ are related as follows: if $\mathbf{v}$ is an eigenvector of $AB$ with $ABv = \lambda \mathbf{v}$, then $B\mathbf{v}$ is an eigenvector of $BA$ with the same eigenvalue:

$$
BA(B\mathbf{v}) = B (AB\mathbf{v}) = B (\lambda \mathbf{v}) = \lambda (B\mathbf{v})
$$

This only fails when $B\mathbf{v} = \mathbf{0}$, but in this case $AB\mathbf{v} = \mathbf{0}$ so $\lambda = 0$. Thus, we can conclude that any non-zero eigenvalues of $AB$ are also eigenvalues of $BA$.

It's much computationally cheaper to compute the eigenvalues of $BA$ (since it's a much smaller matrix), and this gives us all the non-zero eigenvalues of $AB$.
</details>

**Question (hard) - how can you easily compute the SVD of $M$?**

<details>
<summary>Hint</summary>

For a size-$(m, n)$ matrix with $m > n$, the [algorithmic complexity of finding SVD](https://en.wikipedia.org/wiki/Singular_value_decomposition#Numerical_approach) is $O(mn^2)$. So it's relatively cheap to find the SVD of $A$ and $B$ (complexity $mn^2$ vs $m^3$). Can you use that to find the SVD of $M$?
</details>


<details>
<summary>Answer</summary>

It's much cheaper to compute the SVD of the small matrices $A$ and $B$. Denote these SVDs by:

$$
\begin{aligned}
A &= U_A S_A V_A^T \\
B &= U_B S_B V_B^T
\end{aligned}
$$

where $U_A$ and $V_B$ are $(m, n)$, and the other matrices are $(n, n)$.

Then we have:

$$
\begin{aligned}
\quad\quad\quad\quad M &= AB \\
&= U_A (S_A V_A^T U_B S_B) V_B^T
\end{aligned}
$$

Note that the matrix in the middle has size $(n, n)$ (i.e. small), so we can compute its SVD cheaply:

$$
\begin{aligned}
\; S_A V_A^T U_B S_B &= U' S' {V'}^T \quad\quad\quad\quad\quad
\end{aligned}
$$

and finally, this gives us the SVD of $M$:

$$
\begin{aligned}
\quad\quad M &= U_A U' S' {V'}^T V_B^T \\
&= U S {V'}^T
\end{aligned}
$$

where $U = U_A U'$, $V = V_B V'$, and $S = S'$.

All our SVD calculations and matrix multiplications had complexity at most $O(mn^2)$, which is much better than $O(m^3)$ (remember that we don't need to compute all the values of $U = U_A U'$, only the ones which correspond to non-zero singular values).
</details>

If you're curious, you can go to the `FactoredMatrix` documentation to see the implementation of the SVD calculation, as well as other properties and operations.

Now that we've discussed some of the motivations behind having a `FactoredMatrix` class, let's see it in action.

### Basic Examples

We can use the basic class directly - let's make a factored matrix directly and look at the basic operations:

We can also look at the eigenvalues and singular values of the matrix. Note that, because the matrix is rank 2 but 5 by 5, the final 3 eigenvalues and singular values are zero - the factored class omits the zeros.

<details>
<summary>Aside - the sizes of objects returned by the SVD method.</summary>

If $M = USV^T$, and `M.shape = (m, n)` and the rank is `r`, then the SVD method returns the matrices $U, S, V$. They have shape `(m, r)`, `(r,)`, and `(n, r)` respectively, because:

* We don't bother storing the off-diagonal entries of $S$, since they're all zero.
* We don't bother storing the columns of $U$ and $V$ which correspond to zero singular values, since these won't affect the value of $USV^T$.
</details>

We can multiply a factored matrix with an unfactored matrix to get another factored matrix (as in example below). We can also multiply two factored matrices together to get another factored matrix.

If we want to collapse this back to an unfactored matrix, we can use the `AB` property to get the product:

## Reverse-engineering circuits

Within our induction circuit, we have four individual circuits: the OV and QK circuits in our previous token head, and the OV and QK circuits in our induction head. In the following sections of the exercise, we'll reverse-engineer each of these circuits in turn.

* In the section **OV copying circuit**, we'll look at the layer-1 OV circuit.
* In the section **QK prev-token circuit**, we'll look at the layer-0 QK circuit.
* The third section (**K-composition**) is a bit trickier, because it involves looking at the composition of the layer-0 OV circuit **and** layer-1 QK circuit. We will have to do two things:
    1. Show that these two circuits are composing (i.e. that the output of the layer-0 OV circuit is the main determinant of the key vectors in the layer-1 QK circuit).
    2. Show that the joint operation of these two circuits is "make the second instance of a token attend to the token *following* an earlier instance.

The dropdown below contains a diagram explaining how the three sections relate to the different components of the induction circuit. You might have to open it in a new tab to see it clearly.

<details>
<summary>Diagram</summary>

![kcomp](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described_2_new.png)
</details>

After this, we'll have a look at composition scores, which are a more mathematically justified way of showing that two attention heads are composing (without having to look at their behaviour on any particular class of inputs, since it is a property of the actual model weights).

## [1] OV copying circuit

Let's start with an easy part of the circuit — the copying OV circuit of `1.4` and `1.10`. The only interpretable (read: **privileged basis**) things here are the input tokens and output logits, so we want to study the matrix:

$$W_E \cdot W_{OV}^{1.4} \cdot W_U$$

This is the $(d_{\text{vocab}}, d_{\text{vocab}})$-shape matrix that combines with the attention pattern to get us from input to output.

We want to calculate this matrix, and inspect it. We should find that its diagonal values are very high, and its non-diagonal values are very low — in other words, it's approximately the identity. This makes sense — the OV circuit of an induction head should be a **copying** circuit: it copies the value of the source token to the output logits.

### Exercise - compute OV circuit for `1.4`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to ~10 minutes on this exercise.
> ```

Compute the full OV circuit $W_E \cdot W_V^{1.4} \cdot W_O^{1.4} \cdot W_U$ using explicit matrix multiplication.

In [29]:
head_index = 4
layer = 1

W_O = model._model.blocks[layer].attn.W_O[head_index]  # [d_head, d_model]
W_V = model._model.blocks[layer].attn.W_V[head_index]  # [d_model, d_head]
W_E = model._model.embed.weight  # [d_vocab, d_model]
W_U = model._model.unembed.weight.T  # [d_model, d_vocab]

# OV circuit: W_V @ W_O -> [d_model, d_model]
OV_circuit = W_V @ W_O  # [d_model, d_model]
# Full circuit: W_E @ OV @ W_U -> [d_vocab, d_vocab]
full_OV_circuit = W_E @ OV_circuit @ W_U  # [d_vocab, d_vocab]


In [30]:
indices = t.randint(0, model._model.d_vocab, (200,))
full_OV_circuit_sample = full_OV_circuit[indices][:, indices]

imshow(
    to_numpy(full_OV_circuit_sample),
    labels={"x": "Logits on output token", "y": "Input token"},
    title="Full OV circuit for copying head",
    width=700,
    height=600,
)


Compute the full OV circuit as a simple matrix product.

Remember, you can access the model's weights directly e.g. using `model._model.embed.weight` for `W_E`, `model._model.unembed.weight.T` for `W_U`, or `model._model.blocks[layer].attn.W_Q[head]` for per-head weight matrices.

<details>
<summary>Help - I'm not sure how to use this class to compute a product of more than 2 matrices.</summary>

You can compute the OV circuit as a simple matrix product:

```python
OV_circuit = W_V @ W_O  # [d_model, d_model]
full_OV_circuit = W_E @ OV_circuit @ W_U  # [d_vocab, d_vocab]
```
</details>


<details><summary>Solution</summary>

```python
head_index = 4
layer = 1

W_O = model._model.blocks[layer].attn.W_O[head_index]  # [d_head, d_model]
W_V = model._model.blocks[layer].attn.W_V[head_index]  # [d_model, d_head]
W_E = model._model.embed.weight  # [d_vocab, d_model]
W_U = model._model.unembed.weight.T  # [d_model, d_vocab]

OV_circuit = W_V @ W_O  # [d_model, d_model]
full_OV_circuit = W_E @ OV_circuit @ W_U  # [d_vocab, d_vocab]
```
</details>

Now we want to check that this matrix is the identity. Since it's in factored matrix form, this is a bit tricky, but there are still things we can do.

First, to validate that it looks diagonal-ish, let's pick 200 random rows and columns and visualise that - it should at least look identity-ish here! We're using the indexing method of the `FactoredMatrix` class - you can index into it before returning the actual `.AB` value, to avoid having to compute the whole thing (we take advantage of the fact that `A[left_indices, :] @ B[:, right_indices]` is the same as `(A @ B)[left_indices, right_indices]`).

<details>
<summary>Aside - indexing factored matrices</summary>

Yet another nice thing about factored matrices is that you can evaluate small submatrices without having to compute the entire matrix. This is based on the fact that the `[i, j]`-th element of matrix `AB` is `A[i, :] @ B[:, j]`.
</details>

### Exercise - compute circuit accuracy

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
>
> You should spend approximately 10-15 minutes on this exercise.
> ```

When you index a factored matrix, you get back another factored matrix. So rather than explicitly calculating `A[left_indices, :] @ B[:, left_indices]`, we can just write `AB[left_indices, left_indices]`.

You should observe a pretty distinct diagonal pattern here, which is a good sign. However, the matrix is pretty noisy so it probably won't be exactly the identity. Instead, we should come up with a summary statistic to capture a rough sense of "closeness to the identity".

**Accuracy** is a good summary statistic - what fraction of the time is the largest logit on the diagonal? Even if there's lots of noise, you'd probably still expect the largest logit to be on the diagonal a good deal of the time.

If you're on a Colab or have a powerful GPU, you should be able to compute the full matrix and perform this test. However, it's better practice to iterate through this matrix when we can, so that we avoid CUDA issues. We've given you a `batch_size` argument in the function below, and you should try to only explicitly calculate matrices of size `batch_size * d_vocab` rather than the massive matrix of `d_vocab * d_vocab`.

Compute the fraction of the time that the maximum logit is on the diagonal (i.e., the model copies the correct token).

In [31]:
def top_1_acc(full_OV_circuit: Tensor, batch_size: int = 1000) -> float:
    """
    Return the fraction of the time that the maximum value is on the circuit diagonal.
    """
    total = 0

    for indices in t.split(t.arange(full_OV_circuit.shape[0], device=device), batch_size):
        AB_slice = full_OV_circuit[indices]
        total += (t.argmax(AB_slice, dim=1) == indices).float().sum().item()

    return total / full_OV_circuit.shape[0]


if MAIN:
    print(
        f"Fraction of time that the best logit is on diagonal: {top_1_acc(full_OV_circuit):.4f}"
    )


Fraction of time that the best logit is on diagonal: 0.3079


<details>
<summary>Help - I'm not sure whether to take the argmax over rows or columns.</summary>

The OV circuit is defined as `W_E @ W_OV @ W_U`. We can see the i-th row `W_E[i] @ W_OV @ W_U` as the vector representing **the logit vector added at any token which attends to the `i`-th token**, via the attention head with OV matrix `W_OV`.

So we want to take the argmax over rows (i.e. over `dim=1`), because we're interested in the number of tokens `tok` in the vocabulary such that when `tok` is attended to, it is also the top prediction. 

</details>

<details>
<summary>Solution</summary>

```python
def top_1_acc(full_OV_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    """
    Return the fraction of the time that the maximum value is on the circuit diagonal.
    """
    total = 0

    for indices in t.split(t.arange(full_OV_circuit.shape[0], device=device), batch_size):
        AB_slice = full_OV_circuit[indices].AB
        total += (t.argmax(AB_slice, dim=1) == indices).float().sum().item()

    return total / full_OV_circuit.shape[0]
```

</details>

This should return about 30.79% - pretty underwhelming. It goes up to 47.73% for top-5, but still not great. What's up with that?

### Exercise - compute effective circuit

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> ```

Now we return to why we have *two* induction heads. If both have the same attention pattern, the effective OV circuit is actually $W_E(W_V^{1.4}W_O^{1.4}+W_V^{1.10}W_O^{1.10})W_U$. So let's re-run our analysis on this!

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/effective_ov_circuit.png" width="650">

<details>
<summary>Question - why might the model want to split the circuit across two heads?</summary>

Because $W_V W_O$ is a rank 64 matrix. The sum of two is a rank 128 matrix. This can be a significantly better approximation to the desired 50K x 50K matrix!
</details>

In [32]:
# Combined OV circuit for heads 4 and 10
W_V_4 = model._model.blocks[1].attn.W_V[4]  # [d_model, d_head]
W_O_4 = model._model.blocks[1].attn.W_O[4]  # [d_head, d_model]
W_V_10 = model._model.blocks[1].attn.W_V[10]
W_O_10 = model._model.blocks[1].attn.W_O[10]

W_V_both = t.cat([W_V_4, W_V_10], dim=-1)  # [d_model, 2*d_head]
W_O_both = t.cat([W_O_4, W_O_10], dim=0)  # [2*d_head, d_model]

W_OV_eff = W_E @ (W_V_both @ W_O_both) @ W_U

print(
    f"Fraction of the time that the best logit is on the diagonal: {top_1_acc(W_OV_eff):.4f}"
)


Fraction of the time that the best logit is on the diagonal: 0.9556


<details>
<summary>Question - why might the model want to split the circuit across two heads?</summary>

Because $W_V W_O$ is a rank 64 matrix. The sum of two is a rank 128 matrix. This can be a significantly better approximation to the desired 50K x 50K matrix!
</details>

<details>
<summary>Expected output</summary>

You should get an accuracy of 95.6% for top-1 - much better!

Note that you can also try top 5 accuracy, which improves your result to 98%.

</details>


<details><summary>Solution</summary>

```python
W_V_4 = model._model.blocks[1].attn.W_V[4]  # [d_model, d_head]
W_O_4 = model._model.blocks[1].attn.W_O[4]  # [d_head, d_model]
W_V_10 = model._model.blocks[1].attn.W_V[10]
W_O_10 = model._model.blocks[1].attn.W_O[10]

W_V_both = t.cat([W_V_4, W_V_10], dim=-1)  # [d_model, 2*d_head]
W_O_both = t.cat([W_O_4, W_O_10], dim=0)  # [2*d_head, d_model]

W_OV_eff = W_E @ (W_V_both @ W_O_both) @ W_U

print(f"Fraction of the time that the best logit is on the diagonal: {top_1_acc(W_OV_eff):.4f}")
```
</details>

## [2] QK prev-token circuit

The other easy circuit is the QK-circuit of L0H7 — how does it know to be a previous token circuit?

We can multiply out the full QK circuit via the positional embeddings:

$$W_{\text{pos}} \cdot W_Q^{0.7} \cdot (W_K^{0.7})^T \cdot W_{\text{pos}}^T$$

to get a matrix `pos_by_pos` of shape `[max_ctx, max_ctx]`.

### Exercise - interpret full QK-circuit for `0.7`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> ```

Compute and visualize the positional QK circuit for head 0.7.

In [33]:
layer = 0
head_index = 7

W_pos = model._model.pos_embed.weight  # [n_ctx, d_model]
W_Q = model._model.blocks[layer].attn.W_Q[head_index]  # [d_model, d_head]
W_K = model._model.blocks[layer].attn.W_K[head_index]  # [d_model, d_head]
W_QK = W_Q @ W_K.T  # [d_model, d_model]
pos_by_pos_scores = W_pos @ W_QK @ W_pos.T  # [n_ctx, n_ctx]

# Mask, scale and softmax
mask = t.tril(t.ones_like(pos_by_pos_scores)).bool()
pos_by_pos_pattern = t.where(
    mask, pos_by_pos_scores / model._model.d_head**0.5, -1.0e6
).softmax(-1)

print(f"Avg lower-diagonal value: {pos_by_pos_pattern.diag(-1).mean():.4f}")
imshow(
    to_numpy(pos_by_pos_pattern[:200, :200]),
    labels={"x": "Key", "y": "Query"},
    title="Attention patterns for prev-token QK circuit, first 200 indices",
    width=700,
    height=600,
)


Avg lower-diagonal value: 0.9978


### Exercise - interpret full QK-circuit for `0.7`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

The code below plots the full QK circuit for head `0.7` (including a scaling and softmax step, which is meant to mirror how the QK bilinear form will be used in actual attention layers). You should run the code, and interpret the results in the context of the induction circuit.

## [3] K-composition circuit

We now dig into the hard part of the circuit — demonstrating the K-Composition between the previous token head and the induction head.

### Splitting activations

The QK-input for layer 1 is the sum of 14 terms (2 + n_heads) — the token embedding, the positional embedding, and the results of each layer 0 head. So for each head H in layer 1, the attention scores decompose as a sum of terms from each of these components.

### Exercise - analyse the relative importance

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> ```

We can analyse the relative importance of these 14 terms by computing query and key vectors from each component separately.

### Exercise - analyse the relative importance

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 15-25 minutes on these exercises.
> Most of these functions just involve indexing and einsums, but conceptual understanding / figuring out exactly what the question is asking for is the hard part!
> ```

We can now analyse the relative importance of these 14 terms! A very crude measure is to take the norm of each term (by component and position).

Note that this is a pretty dodgy metric - q and k are not inherently interpretable! But it can be a good and easy-to-compute proxy.

<details>
<summary>Question - why are Q and K not inherently interpretable? Why might the norm be a good metric in spite of this?</summary>

They are not inherently interpretable because they operate on the residual stream, which doesn't have a **privileged basis**. You could stick a rotation matrix $R$ after all of the $Q$, $K$ and $V$ weights (and stick a rotation matrix before everything that writes to the residual stream), and the model would still behave exactly the same.

The reason taking the norm is still a reasonable thing to do is that, despite the individual elements of these vectors not being inherently interpretable, it's still a safe bet that if they are larger than they will have a greater overall effect on the residual stream. So looking at the norm doesn't tell us how they work, but it does indicate which ones are more important.
</details>

Fill in the functions below:

In [34]:
def decompose_qk_input(
    model: StandardizedTransformer, tokens: Tensor
) -> Float[Tensor, "n_heads+2 posn d_model"]:
    """
    Retrieves all the input tensors to the first attention layer, and concatenates them along
    the 0th dim.

    The [i, :, :]th element is y_i. The sum of these along dim 0 should be the input to layer 1's attention.
    Components: [embed, pos_embed, head_0_result, head_1_result, ..., head_11_result]
    """
    raw = model._model

    with model.trace(tokens):
        embed_out = model.embed.output.save()          # [batch, seq, d_model]
        pos_embed_out = model.pos_embed.output.save()   # [seq, d_model]
        pattern0 = model.attention_probabilities[0].save()  # [batch, n_heads, seq, seq]

    embed = embed_out.squeeze(0)  # [seq, d_model]
    inp = embed + pos_embed_out   # [seq, d_model]

    # Per-head results from layer 0: V then z @ W_O per head
    attn = raw.blocks[0].attn
    v = t.einsum("sd,hdi->shi", inp, attn.W_V) + attn.b_V  # [seq, n_heads, d_head]
    z = pattern0.squeeze(0) @ v.permute(1, 0, 2)  # [n_heads, seq, d_head]
    head_results = t.einsum("nsh,nhm->nsm", z, attn.W_O)  # [n_heads, seq, d_model]

    y0 = embed.unsqueeze(0)        # [1, seq, d_model]
    y1 = pos_embed_out.unsqueeze(0)  # [1, seq, d_model]

    return t.concat([y0, y1, head_results], dim=0)


def decompose_q(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: StandardizedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of query vectors for each decomposed QK input.
    """
    W_Q = model._model.blocks[1].attn.W_Q[ind_head_index]
    return einops.einsum(
        decomposed_qk_input, W_Q, "n seq d_model, d_model d_head -> n seq d_head"
    )


def decompose_k(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: StandardizedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of key vectors for each decomposed QK input.
    """
    W_K = model._model.blocks[1].attn.W_K[ind_head_index]
    return einops.einsum(
        decomposed_qk_input, W_K, "n seq d_model, d_model d_head -> n seq d_head"
    )


In [35]:
seq_len = 50
batch_size = 1
rep_tokens_1 = generate_repeated_tokens(model, seq_len, batch_size)

ind_head_index = 4

decomposed_qk_input = decompose_qk_input(model, rep_tokens_1)
decomposed_q = decompose_q(decomposed_qk_input, ind_head_index, model)
decomposed_k = decompose_k(decomposed_qk_input, ind_head_index, model)

component_labels = ["Embed", "PosEmbed"] + [
    f"0.{h}" for h in range(model._model.n_heads)
]
for decomposed_input, name in [
    (decomposed_q, "query"),
    (decomposed_k, "key"),
]:
    imshow(
        to_numpy(decomposed_input.pow(2).sum([-1])),
        labels={"x": "Position", "y": "Component"},
        title=f"Norms of components of {name}",
        y=component_labels,
        width=800,
        height=400,
    )


### Exercise - decompose attention scores

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You shouldn't spend more than 5-10 minutes on this exercise.
> Having already done the previous exercises, this one should be easier.
> ```

This tells us which heads are probably important, but we can do better than that. Rather than looking at the query and key components separately, we can see how they combine together - i.e. take the decomposed attention scores.

This is a bilinear function of q and k, and so we will end up with a `decomposed_scores` tensor with shape `[query_component, key_component, query_pos, key_pos]`, where summing along BOTH of the first axes will give us the original attention scores (pre-mask).

Implement the function giving the decomposed scores (remember to scale by `sqrt(d_head)`!) For now, don't mask it.

<details>
<summary>Question - why do I focus on the attention scores, not the attention pattern? (i.e. pre softmax not post softmax)</summary>

Because the decomposition trick *only* works for things that are linear - softmax isn't linear and so we can no longer consider each component independently.
</details>

<details>
<summary>Help - I don't understand what we're doing / why we're doing it.</summary>

Remember that each of our components writes to the residual stream separately. So after layer 0, we have:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/components.png" width="650">

We're particularly interested in the attention scores computed in head `1.4`, and how they depend on the inputs into that head. We've already decomposed the residual stream value $x$ into its terms $e$, $pe$, and $x^ 0$ through $x^{11}$ (which we've labelled $y_0, ..., y_{13}$ for simplicity), and we've done the same for key and query terms. We can picture these terms being passed into head `1.4` as:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/components-2.png" width="650">

So when we expand `attn_scores` out in full, they are a sum of $14^2 = 196$ terms - one for each combination of `(query_component, key_component)`.

---

#### Why is this decomposition useful?

We have a theory about a particular circuit in our model. We think that head `1.4` is an induction head, and the most important components that feed into this head are the prev token head `0.7` (as key) and the token embedding (as query). This is already supported by the evidence of our magnitude plots above (because we saw that `0.7` as key and token embeddings as query were large), but we still don't know how this particular key and query work **together**; we've only looked at them separately.

By decomposing `attn_scores` like this, we can check whether the contribution from combination `(query=tok_emb, key=0.7)` is indeed producing the characteristic induction head pattern which we've observed (and the other 195 terms don't really matter).
</details>

<details>
<summary>What you should see</summary>

You should see that the most important query components are the token and positional embeddings. The most important key components are those from head `0.7`.
</details>

This tells us which heads are probably important, but we can do better. Rather than looking at the query and key components separately, we can see how they combine together — i.e. take the decomposed attention scores.

### Exercise - decompose attention scores

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> ```

In [36]:
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
    model: StandardizedTransformer,
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    """
    Output is decomposed_scores with shape [query_component, key_component, query_pos, key_pos]
    """
    return (
        einops.einsum(
            decomposed_q,
            decomposed_k,
            "q_comp q_pos d_head, k_comp k_pos d_head -> q_comp k_comp q_pos k_pos",
        )
        / (model._model.d_head**0.5)
    )


In [37]:
decomposed_scores = decompose_attn_scores(decomposed_q, decomposed_k, model)

q_label = "Embed"
k_label = "0.7"
decomposed_scores_from_pair = decomposed_scores[
    component_labels.index(q_label), component_labels.index(k_label)
]

imshow(
    to_numpy(t.tril(decomposed_scores_from_pair)),
    title=f"Attention score contributions from query = {q_label}, key = {k_label}<br>(by query & key sequence positions)",
    width=700,
)

decomposed_stds = einops.reduce(
    decomposed_scores,
    "query_decomp key_decomp query_pos key_pos -> query_decomp key_decomp",
    t.std,
)
imshow(
    to_numpy(decomposed_stds),
    labels={"x": "Key Component", "y": "Query Component"},
    title="Std dev of attn score contributions across sequence positions<br>(by query & key comp)",
    x=component_labels,
    y=component_labels,
    width=700,
)


<details>
<summary>Help - I don't understand the interpretation of these plots.</summary>

The first plot tells you that the term $e W_{QK}^{1.4} (x^{0.7})^T$ (i.e. the component of the attention scores for head `1.4` where the query is supplied by the token embeddings and the key is supplied by the output of head `0.7`) produces the distinctive attention pattern we see in the induction head: a strong diagonal stripe.

The second plot shows us the standard deviation of each component pair's contribution to the attention scores. The (Embed, 0.7) pair stands out as having the highest standard deviation, confirming that this is the most important interaction for implementing the induction pattern.
</details>

<details><summary>Solution</summary>

```python
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
    model: StandardizedTransformer,
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    """
    Output is decomposed_scores with shape [query_component, key_component, query_pos, key_pos]

    The [i, j, 0, 0]th element is y_i @ W_QK @ y_j^T (so the sum along both first axes are the
    attention scores)
    """
    return einops.einsum(
        decomposed_q,
        decomposed_k,
        "q_comp q_pos d_head, k_comp k_pos d_head -> q_comp k_comp q_pos k_pos",
    ) / (model._model.d_head**0.5)
```
</details>

### Interpreting the full circuit

Now we know that head `1.4` is composing with head `0.7` via K composition, we can multiply through to create a full circuit:

$$
W_E\, W_{QK}^{1.4}\, (W_{OV}^{0.7})^T\, W_E^T
$$

and verify that it's the identity. (Note, when we say identity here, we're again thinking about it as a distribution over logits, so this should be taken to mean "high diagonal values", and we'll be using our previous metric of `top_1_acc`.)

#### Question - why should this be the identity?

<details>
<summary>Answer</summary>

This matrix is a bilinear form. Its diagonal elements $(A, A)$ are:

$$
A^T \, W_E\, W_{QK}^{1.4}\, W_{OV}^{0.7}\, W_E^T \, A = \underbrace{(A^T W_E W_Q^{1.4})}_{\text{query}} \underbrace{(A^T W_E W_{OV}^{0.7} W_K^{1.4})^T}_{\text{key}}
$$

Intuitively, the query is saying **"I'm looking for a token which followed $A$"**, and the key is saying **"I *am* a token which folllowed $A$"** (recall that $A^T W_E W_{OV}^{0.7}$ is the vector which gets moved one position forward by our prev token head `0.7`).

Now, consider the off-diagonal elements $(A, X)$ (for $X \neq A$). We expect these to be small, because the key doesn't match the query:

$$
A^T \, W_E\, W_{QK}^{1.4}\, W_{OV}^{0.7}\, W_E^T \, X = \underbrace{(\text{I'm looking for a token which followed A})}_{\text{query}} \boldsymbol{\cdot} \underbrace{(\text{I am a token which followed X})}_{\text{key}}
$$


Hence, we expect this to be the identity.

An illustration:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described-K-last.png" width="700">
</details>

Once these tests have passed, you can plot the results:

### Interpreting the full circuit

Now we know that head `1.4` is composing with head `0.7` via K composition, we can multiply through to create a full circuit:

$$W_E\, W_{QK}^{1.4}\, (W_{OV}^{0.7})^T\, W_E^T$$

and verify that it's the identity (high diagonal values).

### Exercise - compute the K-comp circuit

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> ```

### Exercise - compute the K-comp circuit

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-20 minutes on this exercise.
> ```

Calculate the matrix above, as a `FactoredMatrix` object.

<details>
<summary>Aside about multiplying FactoredMatrix objects together.</summary>

If  `M1 = A1 @ B1` and `M2 = A2 @ B2` are factored matrices, then `M = M1 @ M2` returns a new factored matrix. This might be:

```python
FactoredMatrix(M1.AB @ M2.A, M2.B)
```

or it might be:

```python
FactoredMatrix(M1.A, M1.B @ M2.AB)
```

with these two objects corresponding to the factorisations $M = (A_1 B_1 A_2) (B_2)$ and $M = (A_1) (B_1 A_2 B_2)$ respectively.

Which one gets returned depends on the size of the hidden dimension, e.g. `M1.mdim < M2.mdim` then the factorisation used will be $M = A_1 B_1 (A_2 B_2)$.

Remember that both these factorisations are valid, and will give you the exact same SVD. The only reason to prefer one over the other is for computational efficiency (we prefer a smaller bottleneck dimension, because this determines the computational complexity of operations like finding SVD).
</details>

In [38]:
def find_K_comp_full_circuit(
    model: StandardizedTransformer, prev_token_head_index: int, ind_head_index: int
) -> tuple[Tensor, Tensor]:
    """
    Returns (Q_factor, K_factor) tensors that when multiplied give the full K-composition circuit.
    Q_factor: [d_vocab, d_head]
    K_factor: [d_head, d_vocab]
    The full circuit is Q_factor @ K_factor = [d_vocab, d_vocab]
    """
    raw = model._model
    W_E = raw.embed.weight  # [d_vocab, d_model]
    W_Q = raw.blocks[1].attn.W_Q[ind_head_index]  # [d_model, d_head]
    W_K = raw.blocks[1].attn.W_K[ind_head_index]  # [d_model, d_head]
    W_O = raw.blocks[0].attn.W_O[prev_token_head_index]  # [d_head, d_model]
    W_V = raw.blocks[0].attn.W_V[prev_token_head_index]  # [d_model, d_head]

    Q = W_E @ W_Q  # [d_vocab, d_head]
    K = W_E @ W_V @ W_O @ W_K  # [d_vocab, d_head]
    return Q, K.T  # Q: [d_vocab, d_head], K.T: [d_head, d_vocab]


<details><summary>Solution</summary>

```python
def find_K_comp_full_circuit(
    model: StandardizedTransformer, prev_token_head_index: int, ind_head_index: int
) -> tuple[Tensor, Tensor]:
    """
    Returns (Q_factor, K_factor) tensors that when multiplied give the full K-composition circuit.
    Q_factor: [d_vocab, d_head]
    K_factor: [d_head, d_vocab]
    The full circuit is Q_factor @ K_factor = [d_vocab, d_vocab]
    """
    raw = model._model
    W_E = raw.embed.weight  # [d_vocab, d_model]
    W_Q = raw.blocks[1].attn.W_Q[ind_head_index]  # [d_model, d_head]
    W_K = raw.blocks[1].attn.W_K[ind_head_index]  # [d_model, d_head]
    W_O = raw.blocks[0].attn.W_O[prev_token_head_index]  # [d_head, d_model]
    W_V = raw.blocks[0].attn.W_V[prev_token_head_index]  # [d_model, d_head]

    Q = W_E @ W_Q  # [d_vocab, d_head]
    K = W_E @ W_V @ W_O @ W_K  # [d_vocab, d_head]
    return Q, K.T  # Q: [d_vocab, d_head], K.T: [d_head, d_vocab]
```
</details>

You can also try this out for our other induction head `ind_head_index=10`, which should also return a relatively high result. Is it higher than for head `1.4` ?

<details>
<summary>Note - unlike last time, it doesn't make sense to consider the "effective circuit" formed by adding together the weight matrices for heads <code>1.4</code> and <code>1.10</code>. Can you see why?</summary>

Because the weight matrices we're dealing with here are from the QK circuit, not the OV circuit. These don't get combined in a linear way; instead we take softmax over each head's QK-circuit output individually.
</details>

## Further Exploration of Induction Circuits

I now consider us to have fully reverse engineered an induction circuit - by both interpreting the features and by reverse engineering the circuit from the weights. But there's a bunch more ideas that we can apply for finding circuits in networks that are fun to practice on induction heads, so here's some bonus content - feel free to skip to the later bonus ideas though.

### Composition scores

A particularly cool idea in the paper is the idea of [virtual weights](https://transformer-circuits.pub/2021/framework/index.html#residual-comms), or compositional scores. (Though I came up with it, so I'm deeply biased!). This is used [to identify induction heads](https://transformer-circuits.pub/2021/framework/index.html#analyzing-a-two-layer-model).

The key idea of compositional scores is that the residual stream is a large space, and each head is reading and writing from small subspaces. By default, any two heads will have little overlap between their subspaces (in the same way that any two random vectors have almost zero dot product in a large vector space). But if two heads are deliberately composing, then they will likely want to ensure they write and read from similar subspaces, so that minimal information is lost. As a result, we can just directly look at "how much overlap there is" between the output space of the earlier head and the K, Q, or V input space of the later head.

We represent the **output space** with $W_{OV}=W_V W_O$. Call matrices like this $W_A$.

We represent the **input space** with $W_{QK}=W_Q W_K^T$ (for Q-composition), $W_{QK}^T=W_K  W_Q^T$ (for K-Composition) or $W_{OV}=W_V W_O$ (for V-Composition, of the later head). Call matrices like these $W_B$ (we've used this notation so that $W_B$ refers to a later head, and $W_A$ to an earlier head).

<details>
<summary>Help - I don't understand what motivates these definitions.</summary>

Recall that we can view each head as having three input wires (keys, queries and values), and one output wire (the outputs). The different forms of composition come from the fact that keys, queries and values can all be supplied from the output of a different head.

Here is an illustration which shows the three different cases, and should also explain why we use this terminology. You might have to open it in a new tab to see it clearly.

![composition](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/composition_new.png)

</details>

How do we formalise overlap? This is basically an open question, but a surprisingly good metric is $\frac{\|W_AW_B\|_F}{\|W_B\|_F\|W_A\|_F}$ where $\|W\|_F=\sqrt{\sum_{i,j}W_{i,j}^2}$ is the Frobenius norm, the square root of the sum of squared elements. (If you're dying of curiosity as to what makes this a good metric, you can jump to the section immediately after the exercises below.)

### Exercise - calculate composition scores

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You shouldn't spend more than 15-25 minutes on these exercises.
> Writing a composition score function should be fairly easy. The harder part is getting the right weight matrices in the exercises that come after.
> ```

Let's calculate this metric for all pairs of heads in layer 0 and layer 1 for each of K, Q and V composition and plot it.

We'll start by implementing this using plain old tensors (later on we'll see how this can be sped up using the `FactoredMatrix` class). We also won't worry about batching our calculations yet; we'll just do one matrix at a time.

We've given you tensors `q_comp_scores` etc. to hold the composition scores for each of Q, K and V composition (i.e. the `[i, j]`th element of `q_comp_scores` is the Q-composition score between the output from the `i`th head in layer 0 and the input to the `j`th head in layer 1). You should complete the function `get_comp_score`, and then fill in each of these tensors.

### Exercise - calculate composition scores

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> ```

Let's calculate composition scores for all pairs of heads in layer 0 and layer 1 for each of K, Q and V composition.

The composition score between matrices $W_A$ and $W_B$ is:

$$\text{comp\_score}(W_A, W_B) = \frac{\|W_A W_B\|_F}{\|W_A\|_F \|W_B\|_F}$$

where $\|\cdot\|_F$ denotes the Frobenius norm.

<details>
<summary>Aside — theory behind composition scores</summary>

The key insight is a linear algebra result that the squared Frobenius norm equals the sum of squared singular values. So if $W_A = U_A S_A V_A^T$ and $W_B = U_B S_B V_B^T$, then $\|W_A\|_F = \|S_A\|_F$, $\|W_B\|_F = \|S_B\|_F$ and $\|W_A W_B\|_F = \|S_A V_A^T U_B S_B\|_F$. In some sense, $V_A^T U_B$ represents how aligned the subspaces written to and read from are, and the $S_A$ and $S_B$ terms weight by the importance of those subspaces.
</details>

In [39]:
def get_comp_score(
    W_A: Float[Tensor, "in_A out_A"], W_B: Float[Tensor, "out_A out_B"]
) -> float:
    """
    Return the composition score between W_A and W_B.
    """
    W_A_norm = W_A.pow(2).sum().sqrt()
    W_B_norm = W_B.pow(2).sum().sqrt()
    W_AB_norm = (W_A @ W_B).pow(2).sum().sqrt()

    return (W_AB_norm / (W_A_norm * W_B_norm)).item()


In [40]:
# Get all QK and OV matrices for the 2L model
n_heads = model._model.n_heads

W_QK = t.zeros(2, n_heads, model._model.d_model, model._model.d_model, device=device)
W_OV = t.zeros(2, n_heads, model._model.d_model, model._model.d_model, device=device)

for layer_idx in range(2):
    for head_idx in range(n_heads):
        W_Q = model._model.blocks[layer_idx].attn.W_Q[head_idx]
        W_K = model._model.blocks[layer_idx].attn.W_K[head_idx]
        W_V = model._model.blocks[layer_idx].attn.W_V[head_idx]
        W_O = model._model.blocks[layer_idx].attn.W_O[head_idx]
        W_QK[layer_idx, head_idx] = W_Q @ W_K.T
        W_OV[layer_idx, head_idx] = W_V @ W_O

composition_scores = {
    "Q": t.zeros(n_heads, n_heads).to(device),
    "K": t.zeros(n_heads, n_heads).to(device),
    "V": t.zeros(n_heads, n_heads).to(device),
}

for i in tqdm(range(n_heads)):
    for j in range(n_heads):
        composition_scores["Q"][i, j] = get_comp_score(W_OV[0, i], W_QK[1, j])
        composition_scores["K"][i, j] = get_comp_score(W_OV[0, i], W_QK[1, j].T)
        composition_scores["V"][i, j] = get_comp_score(W_OV[0, i], W_OV[1, j])

for comp_type in ["Q", "K", "V"]:
    plot_comp_scores(model, composition_scores[comp_type], f"{comp_type} Composition Scores")


100%|██████████| 12/12 [00:00<00:00, 244.32it/s]


<details><summary>Solution</summary>

```python
def get_comp_score(W_A: Float[Tensor, "in_A out_A"], W_B: Float[Tensor, "out_A out_B"]) -> float:
    """
    Return the composition score between W_A and W_B.
    """
    W_A_norm = W_A.pow(2).sum().sqrt()
    W_B_norm = W_B.pow(2).sum().sqrt()
    W_AB_norm = (W_A @ W_B).pow(2).sum().sqrt()

    return (W_AB_norm / (W_A_norm * W_B_norm)).item()
```
</details>

Once you've passed the tests, you can fill in all the composition scores. Here you should just use a for loop, iterating over all possible pairs of `W_A` in layer 0 and `W_B` in layer 1, for each type of composition. Later on, we'll look at ways to batch this computation.

<details><summary>Solution</summary>

```python
for i in tqdm(range(n_heads)):
    for j in range(n_heads):
        composition_scores["Q"][i, j] = get_comp_score(W_OV[0, i], W_QK[1, j])
        composition_scores["K"][i, j] = get_comp_score(W_OV[0, i], W_QK[1, j].T)
        composition_scores["V"][i, j] = get_comp_score(W_OV[0, i], W_OV[1, j])
```
</details>

### Exercise - Setting a Baseline

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> ```

To interpret the above graphs we need a baseline! A good one is what the scores look like at initialisation. Make a function that randomly generates a composition score 200 times. This model was initialised with **Kaiming Uniform Initialisation**.

In [41]:
def generate_single_random_comp_score(model: StandardizedTransformer) -> float:
    """
    Write a function which generates a single composition score for random matrices.
    """
    W_A_left = t.empty(model._model.d_model, model._model.d_head)
    W_B_left = t.empty(model._model.d_model, model._model.d_head)
    W_A_right = t.empty(model._model.d_model, model._model.d_head)
    W_B_right = t.empty(model._model.d_model, model._model.d_head)

    for W in [W_A_left, W_B_left, W_A_right, W_B_right]:
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))

    W_A = W_A_left @ W_A_right.T
    W_B = W_B_left @ W_B_right.T

    return get_comp_score(W_A, W_B)


In [42]:
n_samples = 300
comp_scores_baseline = np.zeros(n_samples)
for i in tqdm(range(n_samples)):
    comp_scores_baseline[i] = generate_single_random_comp_score(model)

print("\nMean:", comp_scores_baseline.mean())
print("Std:", comp_scores_baseline.std())

hist(
    comp_scores_baseline,
    nbins=50,
    width=800,
    labels={"x": "Composition score"},
    title="Random composition scores",
)

100%|██████████| 300/300 [00:00<00:00, 473.83it/s]


Mean: 0.03605674092968305
Std: 0.00044783671479165383


In [43]:
baseline = comp_scores_baseline.mean()
for comp_type, comp_scores in composition_scores.items():
    plot_comp_scores(
        model, comp_scores, f"{comp_type} Composition Scores", baseline=baseline
    )


<details>
<summary>Some interesting things to observe:</summary>

The most obvious thing that jumps out (when considered in the context of all the analysis we've done so far) is the K-composition scores. `0.7` (the prev token head) is strongly composing with `1.4` and `1.10` (the two induction heads). This is what we expect, and is a good indication that our composition scores are working as intended.

Another interesting thing to note is that the V-composition scores for heads `1.4` and `1.10` with all other heads in layer 0 are very low. In the context of the induction circuit, this makes sense — the induction heads need to compose via K (to find the right token to attend to), not via V (which would mean the output is a function of what the previous token head attended to, not what the induction head attended to).
</details>

<details><summary>Solution</summary>

```python
def generate_single_random_comp_score(model: StandardizedTransformer) -> float:
    """
    Write a function which generates a single composition score for random matrices
    """
    W_A_left = t.empty(model._model.d_model, model._model.d_head)
    W_B_left = t.empty(model._model.d_model, model._model.d_head)
    W_A_right = t.empty(model._model.d_model, model._model.d_head)
    W_B_right = t.empty(model._model.d_model, model._model.d_head)

    for W in [W_A_left, W_B_left, W_A_right, W_B_right]:
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))

    W_A = W_A_left @ W_A_right.T
    W_B = W_B_left @ W_B_right.T

    return get_comp_score(W_A, W_B)
```
</details>

We can re-plot our above graphs with this baseline set to white. Look for interesting things in this graph!

#### Theory + Efficient Implementation

So, what's up with that metric? The key is a cute linear algebra result that the squared Frobenius norm is equal to the sum of the squared singular values.

<details>
<summary>Proof</summary>

We'll give three different proofs:

---

##### Short sketch of proof

Clearly $\|M\|_F^2$ equals the sum of squared singular values when $M$ is diagonal. The singular values of $M$ don't change when we multiply it by an orthogonal matrix (only the matrices $U$ and $V$ will change, not $S$), so it remains to show that the Frobenius norm also won't change when we multiply $M$ by an orthogonal matrix. But this follows from the fact that the Frobenius norm is the sum of the squared $l_2$ norms of the column vectors of $M$, and orthogonal matrices preserve $l_2$ norms. (If we're right-multiplying $M$ by an orthogonal matrix, then we instead view this as performing orthogonal operations on the row vectors of $M$, and the same argument holds.)

---

##### Long proof

$$
\begin{aligned}
\|M\|_F^2 &= \sum_{ij}M_{ij}^2 \\
&= \sum_{ij}((USV^T)_{ij})^2 \\
&= \sum_{ij}\bigg(\sum_k U_{ik}S_{kk}V_{jk}\bigg)^2 \\
&= \sum_{ijk_1 k_2}S_{k_1 k_1} S_{k_2 k_2} U_{i k_1} U_{i k_2} V_{j k_2} V_{j k_2} \\
&= \sum_{k_1 k_2}S_{k_1 k_1} S_{k_2 k_2} \bigg(\sum_i U_{i k_1} U_{i k_2}\bigg)\bigg(\sum_j V_{j k_2} V_{j k_2}\bigg) \\
\end{aligned}
$$

Each of the terms in large brackets is actually the dot product of columns of $U$ and $V$ respectively. Since these are orthogonal matrices, these terms evaluate to 1 when $k_1=k_2$ and 0 otherwise. So we are left with:

$$
\|M\|_F^2 = \sum_{k}S_{k k}^2
$$

---

##### Cute proof which uses the fact that the squared Frobenius norm $|M|^2$ is the same as the trace of $MM^T$

$$
\|M\|_F^2 = \text{Tr}(MM^T) = \text{Tr}(USV^TVSU^T) = \text{Tr}(US^2U^T) = \text{Tr}(S^2 U^T U) = \text{Tr}(S^2) = \|S\|_F^2
$$

where we used the cyclicity of trace, and the fact that $U$ is orthogonal so $U^TU=I$ (and same for $V$). We finish by observing that $\|S\|_F^2$ is precisely the sum of the squared singular values.
</details>

So if $W_A=U_AS_AV_A^T$, $W_B=U_BS_BV_B^T$, then $\|W_A\|_F=\|S_A\|_F$, $\|W_B\|_F=\|S_B\|_F$ and $\|W_AW_B\|_F=\|S_AV_A^TU_BS_B\|_F$. In some sense, $V_A^TU_B$ represents how aligned the subspaces written to and read from are, and the $S_A$ and $S_B$ terms weights by the importance of those subspaces.

<details>
<summary>Click here, if this explanation still seems confusing.</summary>

$U_B$ is a matrix of shape `[d_model, d_head]`. It represents **the subspace being read from**, i.e. our later head reads from the residual stream by projecting it onto the `d_head` columns of this matrix.

$V_A$ is a matrix of shape `[d_model, d_head]`. It represents **the subspace being written to**, i.e. the thing written to the residual stream by our earlier head is a linear combination of the `d_head` column-vectors of $V_A$.

$V_A^T U_B$ is a matrix of shape `[d_head, d_head]`. Each element of this matrix is formed by taking the dot product of two vectors of length `d_model`:

* $v_i^A$, a column of $V_A$ (one of the vectors our earlier head embeds into the residual stream)
* $u_j^B$, a column of $U_B$ (one of the vectors our later head projects the residual stream onto)

Let the singular values of $S_A$ be $\sigma_1^A, ..., \sigma_k^A$ and similarly for $S_B$. Then:

$$
\|S_A V_A^T U_B S_B\|_F^2 = \sum_{i,j=1}^k (\sigma_i^A \sigma_j^B)^2 \|v^A_i \cdot u^B_j\|_F^2
$$

This is a weighted sum of the squared cosine similarity of the columns of $V_A$ and $U_B$ (i.e. the output directions of the earlier head and the input directions of the later head). The weights in this sum are given by the singular values of both $S_A$ and $S_B$ - i.e. if $v^A_i$ is an important output direction, **and** $u_B^i$ is an important input direction, then the composition score will be much higher when these two directions are aligned with each other.

---

To build intuition, let's consider a couple of extreme examples.

* If there was no overlap between the spaces being written to and read from, then $V_A^T U_B$ would be a matrix of zeros (since every $v_i^A \cdot u_j^B$ would be zero). This would mean that the composition score would be zero.
* If there was perfect overlap, i.e. the span of the $v_i^A$ vectors and $u_j^B$ vectors is the same, then the composition score is large. It is as large as possible when the most important input directions and most important output directions line up (i.e. when the singular values $\sigma_i^A$ and $\sigma_j^B$ are in the same order).
* If our matrices $W_A$ and $W_B$ were just rank 1 (i.e. $W_A = \sigma_A u_A v_A^T$, and $W_B = \sigma_B u_B v_B^T$), then the composition score is $|v_A^T u_B|$, in other words just the cosine similarity of the single output direction of $W_A$ and the single input direction of $W_B$.
</details>

### Exercise - batched composition scores

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
>
> This exercise is optional. It's useful for efficiency but not conceptually important.
> ```

Write an efficient batched version of the composition score calculation.

We can also use this insight to write a more efficient way to calculate composition scores - this is extremely useful if you want to do this analysis at scale! The key is that we know that our matrices have a low rank factorisation, and it's much cheaper to calculate the SVD of a narrow matrix than one that's large in both dimensions. See the [algorithm described at the end of the paper](https://transformer-circuits.pub/2021/framework/index.html#induction-heads:~:text=Working%20with%20Low%2DRank%20Matrices) (search for SVD).

So we can work with the `FactoredMatrix` class. This also provides the method `.norm()` which returns the Frobenium norm. This is also a good opportunity to bring back batching - this will sometimes be useful in our analysis. In the function below, `W_As` and `W_Bs` are both >2D factored matrices (e.g. they might represent the OV circuits for all heads in a particular layer, or across multiple layers), and the function's output should be a tensor of composition scores for each pair of matrices `(W_A, W_B)` in the >2D tensors `(W_As, W_Bs)`.

In [44]:
def get_batched_comp_scores(
    W_As: Tensor, W_Bs: Tensor
) -> Tensor:
    """
    Computes compositional scores between pairs of matrices.

    W_As: [n_A, in, mid]
    W_Bs: [n_B, mid, out]

    Returns: [n_A, n_B] composition scores
    """
    # Compute norms
    W_As_norm = W_As.pow(2).sum(dim=(-2, -1)).sqrt()  # [n_A]
    W_Bs_norm = W_Bs.pow(2).sum(dim=(-2, -1)).sqrt()  # [n_B]

    # Compute products: W_ABs[a, b] = W_As[a] @ W_Bs[b]
    W_ABs = t.einsum("aij,bjk->abik", W_As, W_Bs)
    W_ABs_norm = W_ABs.pow(2).sum(dim=(-2, -1)).sqrt()  # [n_A, n_B]

    return W_ABs_norm / (W_As_norm[:, None] * W_Bs_norm[None, :])


<details>
<summary>Hint</summary>

Suppose `W_As` has shape `(A1, A2, ..., Am, A_in, A_out)` and `W_Bs` has shape `(B1, B2, ..., Bn, B_in, B_out)` (where `A_out == B_in`).

It will be helpful to reshape these two tensors so that:

```python
W_As.shape == (A1*A2*...*Am, 1, A_in, A_out)
W_Bs.shape == (1, B1*B2*...*Bn, B_in, B_out)
```

since we can then multiply them together as `W_As @ W_Bs` (broadcasting will take care of this for us!).

You can use `torch.einsum` with batch dimensions to compute all pairwise products efficiently.
</details>


<details><summary>Solution</summary>

```python
def get_batched_comp_scores(W_As: Tensor, W_Bs: Tensor) -> Tensor:
    """
    Computes compositional scores between pairs of matrices.

    W_As: [n_A, in, mid]
    W_Bs: [n_B, mid, out]

    Returns: [n_A, n_B] composition scores
    """
    W_As_norm = W_As.pow(2).sum(dim=(-2, -1)).sqrt()  # [n_A]
    W_Bs_norm = W_Bs.pow(2).sum(dim=(-2, -1)).sqrt()  # [n_B]

    # Compute products: W_ABs[a, b] = W_As[a] @ W_Bs[b]
    W_ABs = t.einsum("aij,bjk->abik", W_As, W_Bs)
    W_ABs_norm = W_ABs.pow(2).sum(dim=(-2, -1)).sqrt()  # [n_A, n_B]

    return W_ABs_norm / (W_As_norm[:, None] * W_Bs_norm[None, :])
```
</details>

### Targeted Ablations

We can refine the ablation technique to detect composition by looking at the effect of the ablation on the attention pattern of an induction head, rather than the loss.

In [45]:
if MAIN:
    seq_len = 50
    rep_tokens_1 = generate_repeated_tokens(model, seq_len, batch_size=1)

    def ablation_induction_score(
        model: StandardizedTransformer,
        tokens: Tensor,
        prev_head_index: int | None,
        ind_head_index: int,
    ) -> float:
        """
        Takes as input the index of the L0 head and the index of the L1 head, and then runs with
        the previous token head ablated and returns the induction score for the ind_head_index.
        Uses nnterp's attention_probabilities to both ablate and read patterns.
        """
        seq_len_half = (tokens.shape[1] - 1) // 2

        with model.trace(tokens):
            # Ablate layer 0 head by zeroing its attention pattern
            if prev_head_index is not None:
                model.attention_probabilities[0][:, prev_head_index] = 0.0
            # Get layer 1's attention pattern (affected by layer 0 ablation)
            pattern1 = model.attention_probabilities[1].save()

        # Extract induction score for specific head
        induction_score = (
            pattern1[0, ind_head_index].diag(-(seq_len_half - 1)).mean().item()
        )
        return induction_score

    baseline_induction_score = ablation_induction_score(model, rep_tokens_1, None, 4)
    print(f"Induction score for no ablations: {baseline_induction_score:.5f}\n")
    for i in range(model._model.n_heads):
        new_induction_score = ablation_induction_score(model, rep_tokens_1, i, 4)
        induction_score_change = new_induction_score - baseline_induction_score
        print(f"Ablation score change for head {i:02}: {induction_score_change:+.5f}")


Induction score for no ablations: 0.62597

Ablation score change for head 00: +0.01608
Ablation score change for head 01: +0.00829
Ablation score change for head 02: +0.03017
Ablation score change for head 03: -0.00131
Ablation score change for head 04: -0.21412
Ablation score change for head 05: +0.00841
Ablation score change for head 06: +0.00853
Ablation score change for head 07: -0.58576
Ablation score change for head 08: +0.03861
Ablation score change for head 09: -0.03228
Ablation score change for head 10: +0.00925
Ablation score change for head 11: +0.00076


<details>
<summary>Question - what is the interpretation of the results you're getting?</summary>

You should have found that the induction score without any ablations is about 0.68, and that most other heads don't change the induction score by much when they are ablated, except for head 7 which reduces the induction score to nearly zero.

This is another strong piece of evidence that head `0.7` is the prev token head in this induction circuit.
</details>

# 5️⃣ Bonus: Introduction to nnterp

Throughout this notebook, you've been accessing model internals using raw NNsight — directly referencing module paths like `model.transformer.h[0].self_attn.c_attn.output`. This works, but it has a significant downside: **the code is architecture-specific**. If you switch from GPT-2 to LLaMA or Gemma, all your module paths break.

The **[nnterp](https://ndif-team.github.io/nnterp/)** library solves this problem. It provides a `StandardizedTransformer` class that wraps any supported transformer model and gives you a **uniform API** for accessing common components — regardless of the underlying architecture.

## What nnterp provides

| What you did manually | nnterp equivalent |
|---|---|
| `model.transformer.h[i].self_attn.c_attn.output` | `model.attention_input[i]` |
| Manual QKV splitting from `c_attn` | `model.attention_probabilities[i]` |
| `model.transformer.h[i].output[0]` | `model.layers_output[i]` |
| `model.lm_head.output` | `model.lm_head.output` (same) |
| Manual logit lens computation | `logit_lens()` built-in |

## Quick demo

Here's what the same GPT-2 analysis looks like with nnterp:

```python
from nnterp import StandardizedTransformer

# Wrap the same model
st_model = StandardizedTransformer("openai-community/gpt2", device_map="auto")

# Access layer outputs — works the same for GPT-2, LLaMA, Gemma, etc.
with st_model.trace(tokens):
    layer_5_output = st_model.layers_output[5].save()
    attn_probs = st_model.attention_probabilities[0].save()

# logit_lens is built in
from nnterp import logit_lens
# logit_lens(st_model, tokens) gives you the predicted tokens at each layer
```

## Why we used raw NNsight first

We deliberately taught you raw NNsight before introducing nnterp, because:

1. **Understanding the internals matters.** When nnterp gives you `attention_probabilities`, you now know exactly what computation that corresponds to (the softmax of scaled QK dot products).
2. **Debugging requires low-level access.** When something doesn't work, you need to understand the actual model architecture.
3. **Not everything is standardized.** For custom models (like our 2L attention-only model), you need raw NNsight.
4. **Transferable skills.** The patterns you learned (trace context, `.save()`, module access) work for *any* PyTorch model, not just transformers.

## When to use what

- **Raw NNsight**: Custom models, unusual architectures, when you need fine-grained control
- **nnterp**: Standard transformer analysis, when you want portable code across architectures, quick prototyping

In the remaining sections of this course (1.3.2 onwards), we'll primarily use nnterp for its convenience, but you'll always have the option to drop down to raw NNsight when needed.

## Bonus - Looking for Circuits in Real LLMs

A particularly cool application of these techniques is looking for real examples of circuits in large language models. With NNsight and nnterp, you can easily load and analyze any HuggingFace model! Many of the techniques we've been using for our 2L transformer carry over to ones with more layers.

Some fun things you might want to try:

- Look for induction heads in larger models - try repeating all of the steps from above. Do they follow the same algorithm?
- Look for neurons that erase info
    - i.e. having a high negative cosine similarity between the input and output weights
- Try to interpret a position embedding.

<details>
<summary>Positional Embedding Hint</summary>

Look at the singular value decomposition `t.svd` and plot the principal components over position space. High ones tend to be sine and cosine waves of different frequencies.
</details>

- Look for heads with interpretable attention patterns: e.g. heads that attend to the same word (or subsequent word) when given text in different languages, or the most recent proper noun, or the most recent full-stop, or the subject of the sentence, etc.
    - Pick a head, ablate it, and run the model on a load of text with and without the head. Look for tokens with the largest difference in loss, and try to interpret what the head is doing.
- Try replicating some of Kevin's work on indirect object identification.
- Inspired by the [ROME paper](https://rome.baulab.info/), use the causal tracing technique of patching in the residual stream - can you analyse how the network answers different facts?

# Bonus

### Looking for Circuits in Real LLMs

A particularly cool application of these techniques is looking for real examples of circuits in large language models. Many of the techniques we've been using for our 2L transformer carry over to ones with more layers.

Some fun things you might want to try:

- Look for induction heads in larger models — try repeating all of the steps from above. Do they follow the same algorithm?
- Look for neurons that erase info (i.e. having a high negative cosine similarity between the input and output weights)
- Try to interpret a position embedding.
- Look for heads with interpretable attention patterns: e.g. heads that attend to the same word when given text in different languages.

### Training Your Own Toy Models

A fun exercise is training models on the minimal task that'll produce induction heads — predicting the next token in a sequence of random tokens with repeated subsequences.

### Further discussion / investigation

Anthropic has written a post on [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) which goes into much deeper discussion on induction heads. The post is structured around six different points of evidence for the hypothesis that **induction heads are the main source of in-context learning in transformer models**, even large ones.

### Training Your Own Toy Models

A fun exercise is training models on the minimal task that'll produce induction heads - predicting the next token in a sequence of random tokens with repeated subsequences. You can get a small 2L Attention-Only model to do this.

<details>
<summary>Tips</summary>

* Make sure to randomise the positions that are repeated! Otherwise the model can just learn the boring algorithm of attending to fixed positions
* It works better if you *only* evaluate loss on the repeated tokens, this makes the task less noisy.
* It works best with several repeats of the same sequence rather than just one.
* If you do things right, and give it finite data + weight decay, you *should* be able to get it to grok - this may take some hyper-parameter tuning though.
* When I've done this I get weird franken-induction heads, where each head has 1/3 of an induction stripe, and together cover all tokens.
* It'll work better if you only let the queries and keys access the positional embeddings, but *should* work either way.
</details>

### Interpreting Induction Heads During Training

A particularly striking result about induction heads is that they consistently [form very abruptly in training as a phase change](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html#argument-phase-change), and are such an important capability that there is a [visible non-convex bump in the loss curve](https://wandb.ai/mechanistic-interpretability/attn-only/reports/loss_ewma-22-08-24-22-08-00---VmlldzoyNTI2MDM0?accessToken=r6v951q0e1l4q4o70wb2q67wopdyo3v69kz54siuw7lwb4jz6u732vo56h6dr7c2) (in this model, approx 2B to 4B tokens). I have a bunch of checkpoints for this model, you can try re-running the induction head detection techniques on intermediate checkpoints and see what happens. (Bonus points if you have good ideas for how to efficiently send a bunch of 300MB checkpoints from Wandb lol)

### Further discussion / investigation

Anthropic has written a post on [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) which goes into much deeper discussion on induction heads. The post is structured around six different points of evidence for the hypothesis that **induction heads are the main source of in-context learning in transformer models**, even large ones. Briefly, these are:

1. Transformers undergo a "phase change" where they suddenly become much better at in-context learning, and this is around the same time induction heads appear.
2. When we change the transformer's architecture to make it easier for induction heads to form, we get a corresponding improvement in in-context learning.
3. When we ablate induction heads at runtime, in-context learning gets worse.
4. We have specific examples of induction heads performing more complex in-context learning algorithms (you'll have the opportunity to investigate one of these later - **indirect object identification**).
5. We have a mechanistic explanation of induction heads, which suggests natural extensions to more general forms of in-context learning.
6. In-context learning-related behaviour is generally smoothly continuous between small and large models, suggesting that the underlying mechanism is also the same.

Here are a few questions for you:

* How compelling do you find this evidence? Discuss with your partner.
    * Which points do you find most compelling?
    * Which do you find least compelling?
    * Are there any subset of these which would be enough to convince you of the hypothesis, in the absence of others?
* In point 3, the paper observes that in-context learning performance degrades when you ablate induction heads. While we measured this by testing the model's ability to copy a duplicated random sequence, the paper used **in-context learning score** (the loss of the 500th token in the context, minus the loss on the 50th token).
    * Can you see why this is a reasonable metric?
    * Can you replicate these results (maybe on a larger model than the 2-layer one we've been using)?
* In point 4 (more complex forms of in-context learning), the paper suggests the natural extension of "fuzzy induction heads", which match patterns like `[A*][B*]...[A][B]` rather than `[A][B]...[A][B]` (where the `*` indicates some form of linguistic similarity, not necessarily being the same token).
    * Can you think of any forms this might take, i.e. any kinds of similarity which induction heads might pick up on? Can you generate examples?